In [ ]:
from pathlib import Path
import json
import re
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import pearsonr, spearmanr, mannwhitneyu, kruskal


def parse_roles(x):
    """
    Accept:
    
      - list/tuple/set of strings
      - a single string "Role1 Role2 Role3" (space-separated)
      - a string with commas also works
    Return: list[str]
    """
    if x is None:
        return []
    if isinstance(x, (list, tuple, set)):
        return [str(r).strip() for r in x if str(r).strip()]
    s = str(x).strip()
    if not s:
        return []
    # split on commas OR whitespace
    parts = re.split(r"[,\s]+", s)
    return [p.strip() for p in parts if p.strip()]


def safe_float(x):
    try:
        if x is None:
            return np.nan
        s = str(x).strip()
        if s == "" or s.lower() == "nan":
            return np.nan
        return float(s)
    except Exception:
        return np.nan


def safe_json_list(x):
    """
    Parse a JSON list stored as a string; return [] if empty/bad.
    """
    if x is None:
        return []
    s = str(x).strip()
    if s == "" or s.lower() == "nan":
        return []
    try:
        obj = json.loads(s)
        return obj if isinstance(obj, list) else []
    except Exception:
        return []


def sign3(v):
    """Return -1,0,1 based on sign."""
    if v > 0:
        return 1
    if v < 0:
        return -1
    return 0


def choice_to_sign(choice):
    if choice is None:
        return None
    c = str(choice).strip().upper()
    if c in {"P", "POS", "POSITIVE", "+", "+1", "1"}:
        return 1
    if c in {"N", "NEG", "NEGATIVE", "-", "-1"}:
        return -1
    return None


def extract_variant_sign(item, variant_idx):
    # Prefer explicit choice labels
    s = choice_to_sign(item.get(f"choice_v{variant_idx}", None))
    if s in (-1, 1):
        return s

    # Fallback to mapped score / raw score sign
    for key in (f"score_v{variant_idx}_mapped", f"score_v{variant_idx}"):
        v = safe_float(item.get(key, np.nan))
        if np.isfinite(v):
            if v > 0:
                return 1
            if v < 0:
                return -1
    return None


def rescale_to_minus2_plus2(x, conf_ref=100.0):
    """
    Rescale signed probablity to [-2, 2] using a fixed probablity reference.

    probablity in the runner is defined on [0, 100], so fixed scaling keeps
    scores comparable across rows. Row-wise scaling (e.g., by per-row max
    probablity) can inflate low-probablity rows.
    """
    x = safe_float(x)
    conf_ref = safe_float(conf_ref)
    if not np.isfinite(x):
        return np.nan

    if not np.isfinite(conf_ref) or conf_ref <= 0:
        conf_ref = 100.0

    y = 2.0 * (x / conf_ref)
    return float(np.clip(y, -2.0, 2.0))


def parse_ipc_axes(label):
    """
    IPC label format expected: A{P|M|N}C{P|M|N} e.g. APCP, ANCM, AMCN.
    Returns (ipc_A, ipc_C) where P=+1, M=0, N=-1, else (nan,nan).
    """
    if label is None or (isinstance(label, float) and np.isnan(label)):
        return np.nan, np.nan
    s = str(label).strip()
    m = re.match(r"^A([PNM])C([PNM])$", s)
    if not m:
        return np.nan, np.nan
    mp = {"P": 1, "M": 0, "N": -1}
    return mp[m.group(1)], mp[m.group(2)]


def load_question(question_path: Path):
    cfg = json.loads(question_path.read_text(encoding="utf-8"))
    q = cfg["outside_judge"][0]
    return q


def to_long_eval_df(df_wide: pd.DataFrame, roles_of_interest=None) -> pd.DataFrame:
    """
    Convert a wide per-conversation dataframe into long per-evaluated-speaker rows.
    Keeps A and/or B if avg_score_* is present (successful parse).

    Adds probablity-derived secondary score fields when available from evidence items:
      - avg_signed_conf
      - avg_signed_conf_scaled  (rescaled to [-2, 2])
      - has_variant_disagreement
      - disagreement_item_rate
    """
    records = []
    for _, row in df_wide.iterrows():
        for spk in ["a", "b"]:
            avg = safe_float(row.get(f"avg_score_{spk}", ""))
            # "evaluated successfully" = avg_score exists
            if np.isnan(avg):
                continue

            role = row.get(f"role_{spk}", "")
            if roles_of_interest is not None and role not in roles_of_interest:
                continue

            turns = safe_json_list(row.get(f"evidence_{spk}", ""))

            item_disagrees = []
            signed_conf_vals = []
            conf_abs_vals = []
            score_v0_vals = []
            score_v1_vals = []

            for t in turns:
                s0 = extract_variant_sign(t, 0)
                s1 = extract_variant_sign(t, 1)

                v0 = safe_float(t.get("score_v0_mapped", t.get("score_v0", np.nan)))
                v1 = safe_float(t.get("score_v1_mapped", t.get("score_v1", np.nan)))
                if np.isfinite(v0):
                    score_v0_vals.append(float(v0))
                if np.isfinite(v1):
                    score_v1_vals.append(float(v1))

                if s0 in (-1, 1) and s1 in (-1, 1):
                    item_disagrees.append(bool(s0 != s1))

                for vidx, s in [(0, s0), (1, s1)]:
                    # Backward-compatible: support both probablity_v* and prob_v* fields.
                    conf = safe_float(t.get(f"probablity_v{vidx}", np.nan))
                    if not np.isfinite(conf):
                        conf = safe_float(t.get(f"prob_v{vidx}", np.nan))
                    if np.isfinite(conf) and s in (-1, 1):
                        signed_conf_vals.append(float(s * conf))
                        conf_abs_vals.append(float(abs(conf)))

            avg_signed_conf = float(np.mean(signed_conf_vals)) if signed_conf_vals else np.nan
            # Support both probability-scale (0..1) and probablity-scale (0..100).
            if signed_conf_vals:
                max_abs_conf = float(np.nanmax(conf_abs_vals)) if conf_abs_vals else np.nan
                conf_scale_ref = 1.0 if np.isfinite(max_abs_conf) and max_abs_conf <= 1.0 else 100.0
            else:
                conf_scale_ref = np.nan
            avg_signed_conf_scaled = rescale_to_minus2_plus2(avg_signed_conf, conf_scale_ref)

            disagreement_item_rate = float(np.mean(item_disagrees)) if item_disagrees else np.nan
            has_variant_disagreement = bool(any(item_disagrees)) if item_disagrees else False

            row_mean_score_v0 = float(np.mean(score_v0_vals)) if score_v0_vals else np.nan
            row_mean_score_v1 = float(np.mean(score_v1_vals)) if score_v1_vals else np.nan
            row_sign_v0 = sign3(row_mean_score_v0) if np.isfinite(row_mean_score_v0) else 0
            row_sign_v1 = sign3(row_mean_score_v1) if np.isfinite(row_mean_score_v1) else 0
            row_level_variant_disagreement = bool(
                row_sign_v0 in (-1, 1) and row_sign_v1 in (-1, 1) and row_sign_v0 != row_sign_v1
            )

            rec = {
                "prompt_id_unique": row.get("prompt_id_unique", ""),
                "a_id": row.get("a_id", ""),
                "b_id": row.get("b_id", ""),
                "speaker": spk.upper(),
                "role": role,
                "category": row.get(f"category_{spk}", ""),
                "ipc": row.get(f"ipc_{spk}", ""),
                "avg_score": avg,
                "flip_rate": safe_float(row.get(f"flip_rate_{spk}", "")),
                "evidence_turnovers": turns,
                "fail_notes": safe_json_list(row.get(f"fail_note_{spk}", "")),
                "avg_signed_conf": avg_signed_conf,
                "avg_signed_conf_scaled": avg_signed_conf_scaled,
                "conf_scale_ref": conf_scale_ref,
                "n_items_scored": len(turns),
                "disagreement_item_rate": disagreement_item_rate,
                "has_variant_disagreement": has_variant_disagreement,
                "row_mean_score_v0": row_mean_score_v0,
                "row_mean_score_v1": row_mean_score_v1,
                "row_level_variant_disagreement": row_level_variant_disagreement,
            }
            records.append(rec)

    out = pd.DataFrame(records)
    if out.empty:
        return out

    # Add IPC axes
    out[["ipc_A", "ipc_C"]] = out["ipc"].apply(lambda x: pd.Series(parse_ipc_axes(x)))

    # Default predicted pole from avg_score (can be overwritten downstream by score mode).
    out["pred_pole"] = out["avg_score"].apply(sign3)

    return out


def flatten_turnovers(eval_df: pd.DataFrame) -> pd.DataFrame:
    """
    Expand evidence_turnovers into a turnover-level dataframe.
    """
    rows = []
    for _, r in eval_df.iterrows():
        turns = r.get("evidence_turnovers", [])
        for t in turns:
            rows.append({
                "prompt_id_unique": r["prompt_id_unique"],
                "a_id": r.get("a_id", ""),
                "b_id": r.get("b_id", ""),
                "speaker": r["speaker"],
                "role": r["role"],
                "category": r["category"],
                "ipc": r["ipc"],
                "ipc_A": r.get("ipc_A", np.nan),
                "ipc_C": r.get("ipc_C", np.nan),
                "turnover_file": t.get("turnover_file", ""),
                "turnover_direction": t.get("turnover_direction", ""),
                "target_speaker": t.get("target_speaker", np.nan),
                "score_v0": t.get("score_v0", np.nan),
                "score_v1": t.get("score_v1", np.nan),
                "score_avg": t.get("score_avg", np.nan),
                "choice_v0": t.get("choice_v0", None),
                "choice_v1": t.get("choice_v1", None),
                "probablity_v0": t.get("probablity_v0", np.nan),
                "probablity_v1": t.get("probablity_v1", np.nan),
            })
    return pd.DataFrame(rows)



In [ ]:
RUN_ROOT = Path("./runs")   # CLI layout; set to Path("./runs_bpc_evidence") for compatibility outputs
IDX = 1                               # question index folder q{IDX}
SUBFOLDERS = {
    "models/qwen_omni": "Qwen",
    "models/kimi_audio": "Kimi",
    "models/granite_speech": "Granite",
    "models/gpt_audio": "GPT",
    "models/gemini_audio": "Gemini",
}
V3_MODEL_IDS = {
    "Qwen": "qwen-omni",
    "Kimi": "kimi-audio",
    "Granite": "granite-speech",
    "GPT": "gpt-audio",
    "Gemini": "gemini-audio",
    "Qwen transcript ablation": "qwen-transcript-ablation",
}

def _latest_cli_run_dir(project_root: Path, run_root_name: str, model_name: str, idx: int):
    model_id = V3_MODEL_IDS.get(model_name)
    if not model_id:
        return None
    base = project_root / run_root_name / model_id / f"S{idx}"
    if not base.exists():
        return None
    candidates = [p for p in base.iterdir() if (p / "legacy_filtered_subset.csv").exists()]
    if not candidates:
        return None
    return max(candidates, key=lambda p: (p / "legacy_filtered_subset.csv").stat().st_mtime)

def resolve_run_dir(project_root: Path, run_root_name: str, subfolder: str, model_name: str, idx: int) -> Path:
    old_dir = project_root / subfolder / run_root_name / f"q{idx}"
    old_csv = old_dir / f"filtered_subset_{idx}.csv"
    if old_csv.exists():
        return old_dir
    cli_dir = _latest_cli_run_dir(project_root, run_root_name, model_name, idx)
    return cli_dir if cli_dir is not None else old_dir

def resolve_csv_path(project_root: Path, run_root_name: str, subfolder: str, model_name: str, idx: int) -> Path:
    run_dir = resolve_run_dir(project_root, run_root_name, subfolder, model_name, idx)
    cli_csv = run_dir / "legacy_filtered_subset.csv"
    return cli_csv if cli_csv.exists() else run_dir / f"filtered_subset_{idx}.csv"

def resolve_question_path(project_root: Path, run_root_name: str, subfolder: str, model_name: str, idx: int) -> Path:
    run_dir = resolve_run_dir(project_root, run_root_name, subfolder, model_name, idx)
    cli_question = run_dir / "question.json"
    return cli_question if cli_question.exists() else run_dir / f"question_{idx}.json"

SOURCE_INDEX_BY_PUBLIC_IDX = {0: 0, 1: 1, 2: 2, 3: 3, 4: 4, 5: 5, 6: 7, 7: 8, 8: 9}

def load_bundled_question(idx: int):
    source_idx = SOURCE_INDEX_BY_PUBLIC_IDX[int(idx)]
    candidates = [
        Path("stancebench/metadata/questions_main.json"),
        Path("metadata/questions_main.json"),
        Path("../stancebench/metadata/questions_main.json"),
    ]
    for path in candidates:
        if path.exists():
            data = json.loads(path.read_text())
            questions = data.get("outside_judge", data) if isinstance(data, dict) else data
            for question in questions:
                if int(question.get("source_index", question.get("idx", -1))) == source_idx:
                    print(f"[INFO] using bundled StanceBench question metadata from {path} for S{idx}")
                    return question
            raise KeyError(f"Bundled question metadata does not contain source index {source_idx} for S{idx}: {path}")
    raise FileNotFoundError("Could not find bundled StanceBench question metadata: stancebench/metadata/questions_main.json")

# Ablation-only comparison:
# SUBFOLDERS = {"models/qwen_omni": "Qwen", "models/qwen_transcript_ablation": "Qwen transcript ablation"}

filter_by_lowest = True          # True -> keep only intersection of successfully scored rows across models

use_prob = True                    # True -> use probablity-derived secondary score
exclude_variant_disagree = True       # apply disagreement-based row exclusion before downstream analysis
disagree_rate_exclude_thresh = 0.2    # exclude rows with disagreement_item_rate >= this threshold

RUN_DIR = RUN_ROOT / f"q{IDX}"
CSV_PATH = RUN_DIR / f"filtered_subset_{IDX}.csv"
QUESTION_PATH = RUN_DIR / f"question_{IDX}.json"

if IDX == 0:
    POS_ROLES = "Friendly Warm Approachable Welcoming"
    NEG_ROLES = "Aloof Distant Impersonal Indifferent"
elif IDX == 1:
    POS_ROLES = "Empathetic Considerate Understanding Concerned"
    NEG_ROLES = "Insensitive Unsympathetic Inconsiderate Callous"
elif IDX == 2:
    POS_ROLES = "Polite Respectful Courteous"
    NEG_ROLES = "Disrespectful Impolite Uncivil"
elif IDX == 3:
    POS_ROLES = "Assertive Decisive Self-assured Firm"
    NEG_ROLES = "Indecisive Self-doubting Unassertive Timid"
elif IDX == 4:
    POS_ROLES = "Honest Ingenuous Uncalculating"
    NEG_ROLES = "Manipulative Calculating Devious"
elif IDX == 5:
    POS_ROLES = "Alert Attentive Concentrating Engaged"
    NEG_ROLES = "Bewildered Distracted Drowsy Unfocused"
elif IDX == 6:
    POS_ROLES = "Engaging Sociable Gregarious Outgoing"
    NEG_ROLES = "Withdrawn Disengaged Reticent Taciturn"
elif IDX == 7:
    POS_ROLES = "Submissive Meek Yielding Undemanding"
    NEG_ROLES = "Forceful Overbearing Dominant Domineering"
elif IDX == 8:
    POS_ROLES = "Stable Steady Unaggressive Unargumentative"
    NEG_ROLES = "Aggressive Cruel Ruthless Vindictive"
else:
    raise ValueError(f"No POS/NEG roles configured for IDX={IDX}")

# If you want, explicitly define the full role set used in that run:

POS_ROLES = parse_roles(POS_ROLES)
NEG_ROLES = parse_roles(NEG_ROLES)
ROLES_OF_INTEREST = set(POS_ROLES + NEG_ROLES)

print(
    f"filter_by_lowest={filter_by_lowest}, use_prob={use_prob}, "
    f"exclude_variant_disagree={exclude_variant_disagree}, "
    f"disagree_rate_exclude_thresh={disagree_rate_exclude_thresh}"
)
POS_ROLES, NEG_ROLES, ROLES_OF_INTEREST


In [ ]:
_question_path = QUESTION_PATH
if not _question_path.exists():
    # Search both cwd and cwd.parent so this works from the project root
    # and older layouts where model folders are one level up.
    _cwd = Path.cwd().resolve()
    _candidate_roots = [_cwd, _cwd.parent]
    _tried = []

    for _project_root in _candidate_roots:
        for _subfolder in SUBFOLDERS.keys():  # preserves insertion order
            _candidate = resolve_question_path(_project_root, RUN_ROOT.name, _subfolder, SUBFOLDERS[_subfolder], IDX)
            _tried.append(str(_candidate))
            if _candidate.exists():
                _question_path = _candidate
                break
        if _question_path.exists():
            break
    else:
        _question_path = None

q = load_question(_question_path) if _question_path is not None else load_bundled_question(IDX)

question_text = q["question"]
cats = q.get("related_categories", [])
pos_def = q.get("positive_defination", "")
neg_def = q.get("negative_defination", "")

print("QUESTION:")
print(question_text)
print("\nCATEGORIES:", cats)
print("\nPOS DEF:", pos_def)
print("NEG DEF:", neg_def)

# Expect 2 categories for consistency checks (Q9 is tri-category).
if int(IDX) == 9 and len(cats) >= 3:
    POS_CATEGORY = [cats[0], cats[1]]
    NEG_CATEGORY = cats[2]
    POS_CATEGORY_LABEL = f"{cats[0]} + {cats[1]}"
    NEG_CATEGORY_LABEL = str(cats[2])
else:
    POS_CATEGORY = cats[0] if len(cats) >= 1 else None
    NEG_CATEGORY = cats[1] if len(cats) >= 2 else None
    POS_CATEGORY_LABEL = str(cats[0]) if len(cats) >= 1 else "POS"
    NEG_CATEGORY_LABEL = str(cats[1]) if len(cats) >= 2 else "NEG"

if len(cats) != 2 and int(IDX) != 9:
    print("\n[WARNING] related_categories is not length-2. "
          "Category-pole consistency may need manual POS_CATEGORY / NEG_CATEGORY.")


In [ ]:
from collections import OrderedDict

# ---- Multi-model loader ----
# NOTE:
#  - QUESTION_PATH in Cell 2 still points to the compatibility layout.
#    resolve_question_path also supports CLI runs/<model>/S{IDX}/<run_id>/question.json.
#  - Per-model CSVs are resolved from the CLI layout first, then the compatibility layout.

# Preserve insertion order from SUBFOLDERS
MODEL_SPECS = list(SUBFOLDERS.items())  # [(subfolder_name, model_name), ...]

def infer_project_root(run_root: Path, subfolders: dict) -> Path:
    """Infer the directory that contains the per-model subfolders."""
    cwd = Path.cwd().resolve()
    parent = cwd.parent

    # Preferred layout: run from the StanceBench project root where model folders live in cwd.
    if any((cwd / sub).exists() for sub in subfolders):
        return cwd

    # Backward-compatible layout: model folders are in the parent directory.
    if any((parent / sub).exists() for sub in subfolders):
        return parent

    # If RUN_ROOT is absolute and looks like .../<subfolder>/<runs_root>, use that base.
    if run_root.is_absolute():
        return run_root.parent.parent

    # Last resort: stick with cwd.
    return cwd

PROJECT_ROOT = infer_project_root(RUN_ROOT, SUBFOLDERS)
RUN_ROOT_NAME = RUN_ROOT.name

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RUN_ROOT_NAME:", RUN_ROOT_NAME)
print("Analyzing IDX:", IDX)
print("Models configured (subfolder -> model):", SUBFOLDERS)

MODEL_TO_SUBFOLDER = OrderedDict()
MODEL_RUN_ROOTS = OrderedDict()
MODEL_WIDE_DFS = OrderedDict()
MODEL_EVAL_DFS = OrderedDict()
MODEL_EVAL_DFS_RAW = OrderedDict()      # before optional filter_by_lowest and disagreement exclusion
MODEL_EVAL_DFS_PRE = OrderedDict()      # after optional filter_by_lowest, before disagreement exclusion
MODEL_SCORE_COL = OrderedDict()
MODEL_SCORE_LABEL = OrderedDict()
MODEL_PLOT_LABEL = OrderedDict()
MODEL_PLOT_REF_LEVELS = OrderedDict()
MODEL_ITEM_FLIP_RATE_MEAN_PRE = OrderedDict()

for subfolder, model_name in MODEL_SPECS:
    run_root = PROJECT_ROOT / subfolder / RUN_ROOT_NAME
    MODEL_TO_SUBFOLDER[model_name] = subfolder
    MODEL_RUN_ROOTS[model_name] = run_root

# Pass 1: load each model and build per-model successfully-scored rows.
for subfolder, model_name in MODEL_SPECS:
    run_root = MODEL_RUN_ROOTS[model_name]
    run_dir = resolve_run_dir(PROJECT_ROOT, RUN_ROOT_NAME, subfolder, model_name, IDX)
    csv_path = resolve_csv_path(PROJECT_ROOT, RUN_ROOT_NAME, subfolder, model_name, IDX)

    print("\n" + "=" * 110)
    print(f"MODEL: {model_name}    SUBFOLDER: {subfolder}")
    print(f"RUN_DIR: {run_dir}")
    print(f"CSV_PATH: {csv_path}")
    print("=" * 110)

    if not csv_path.exists():
        print(f"[WARN] Missing CSV for model={model_name}: {csv_path}")
        continue

    df_wide = pd.read_csv(csv_path, dtype=str, low_memory=False)

    # Long-form = 1 row per successfully evaluated speaker in this model.
    eval_df = to_long_eval_df(df_wide, roles_of_interest=ROLES_OF_INTEREST)
    eval_df_raw = eval_df.copy()

    # Select analysis score mode
    if use_prob:
        n_conf = int(eval_df["avg_signed_conf_scaled"].notna().sum()) if (len(eval_df) and "avg_signed_conf_scaled" in eval_df.columns) else 0
        if n_conf == 0:
            print("[WARN] use_prob=True but no probablity-derived score found; fallback to avg_score.")
            SCORE_COL = "avg_score"
            SCORE_LABEL = "avg_score (fallback)"
        else:
            SCORE_COL = "avg_signed_conf_scaled"
            SCORE_LABEL = "signed_probablity_scaled"
    else:
        SCORE_COL = "avg_score"
        SCORE_LABEL = "avg_score"

    if SCORE_COL not in eval_df.columns:
        eval_df[SCORE_COL] = np.nan

    eval_df["score_used"] = pd.to_numeric(eval_df[SCORE_COL], errors="coerce")
    eval_df["pred_pole"] = eval_df["score_used"].apply(sign3)

    # Plot-only scaling: when use_prob=True, normalize plotted scores from [-2, 2] to [-1, 1].
    if use_prob:
        eval_df["score_plot"] = pd.to_numeric(eval_df["score_used"], errors="coerce") / 2.0
        PLOT_LABEL = f"{SCORE_LABEL}"
        PLOT_REF_LEVELS = [-1, -0.5, 0, 0.5, 1]
    else:
        eval_df["score_plot"] = pd.to_numeric(eval_df["score_used"], errors="coerce")
        PLOT_LABEL = SCORE_LABEL
        PLOT_REF_LEVELS = [-2, -1, 0, 1, 2]

    MODEL_WIDE_DFS[model_name] = df_wide
    MODEL_EVAL_DFS_RAW[model_name] = eval_df_raw
    MODEL_EVAL_DFS_PRE[model_name] = eval_df
    MODEL_SCORE_COL[model_name] = SCORE_COL
    MODEL_SCORE_LABEL[model_name] = SCORE_LABEL
    MODEL_PLOT_LABEL[model_name] = PLOT_LABEL
    MODEL_PLOT_REF_LEVELS[model_name] = PLOT_REF_LEVELS

MODEL_ORDER = [m for (_, m) in MODEL_SPECS if m in MODEL_EVAL_DFS_PRE]

# Optional fairness filter: keep only the intersection of successfully scored rows.
def _row_key_series(eval_df: pd.DataFrame):
    if len(eval_df) == 0:
        return pd.Series(dtype="object")

    # Preferred stable key shared across model runs of the same subset.
    if {"a_id", "b_id", "speaker"}.issubset(eval_df.columns):
        return pd.Series(
            list(zip(eval_df["a_id"].astype(str), eval_df["b_id"].astype(str), eval_df["speaker"].astype(str))),
            index=eval_df.index,
        )

    # Fallback when a_id/b_id are unavailable.
    if {"prompt_id_unique", "speaker"}.issubset(eval_df.columns):
        return pd.Series(
            list(zip(eval_df["prompt_id_unique"].astype(str), eval_df["speaker"].astype(str))),
            index=eval_df.index,
        )

    return None

if filter_by_lowest and len(MODEL_ORDER) > 0:
    key_sets = []
    for model_name in MODEL_ORDER:
        eval_df = MODEL_EVAL_DFS_PRE[model_name]
        row_keys = _row_key_series(eval_df)
        if row_keys is None:
            key_sets.append(set())
            continue
        key_sets.append(set(row_keys.tolist()))

    common_keys = set.intersection(*key_sets) if key_sets else set()
    print("\n[filter_by_lowest] enabled")
    print("[filter_by_lowest] common successfully scored rows:", len(common_keys))

    for model_name in MODEL_ORDER:
        eval_df = MODEL_EVAL_DFS_PRE[model_name]
        row_keys = _row_key_series(eval_df)
        if row_keys is None:
            MODEL_EVAL_DFS_PRE[model_name] = eval_df.iloc[0:0].copy()
            print(f"[filter_by_lowest] {model_name}: {len(eval_df)} -> 0")
            continue

        keep_mask = row_keys.isin(common_keys)
        MODEL_EVAL_DFS_PRE[model_name] = eval_df.loc[keep_mask].copy()
        print(f"[filter_by_lowest] {model_name}: {len(eval_df)} -> {len(MODEL_EVAL_DFS_PRE[model_name])}")
elif filter_by_lowest:
    print("\n[filter_by_lowest] enabled, but no models were loaded.")

# Pass 2: diagnostics + disagreement filtering from the selected pre-filter set.
for model_name in MODEL_ORDER:
    df_wide = MODEL_WIDE_DFS[model_name]
    eval_df = MODEL_EVAL_DFS_PRE[model_name].copy()
    SCORE_LABEL = MODEL_SCORE_LABEL[model_name]

    # Disagreement / flip diagnostics (before disagreement filtering)
    row_disagree_rate = float(eval_df["has_variant_disagreement"].mean()) if (len(eval_df) and "has_variant_disagreement" in eval_df.columns) else np.nan
    item_flip_rate_mean = float(pd.to_numeric(eval_df["disagreement_item_rate"], errors="coerce").dropna().mean()) if (len(eval_df) and "disagreement_item_rate" in eval_df.columns) else np.nan
    runner_flip_rate_mean = float(pd.to_numeric(eval_df["flip_rate"], errors="coerce").dropna().mean()) if (len(eval_df) and "flip_rate" in eval_df.columns) else np.nan
    MODEL_ITEM_FLIP_RATE_MEAN_PRE[model_name] = item_flip_rate_mean

    print("Conversations (rows in CSV):", len(df_wide))
    print("Evaluated speakers (rows in eval_df, pre-filter):", len(eval_df))
    print("Score mode:", SCORE_LABEL)
    print("Row disagreement rate (any v0/v1 disagreement):", row_disagree_rate)
    print("Mean item disagreement rate:", item_flip_rate_mean)
    print("Mean runner flip_rate:", runner_flip_rate_mean)

    if exclude_variant_disagree and len(eval_df):
        before_eval = len(eval_df)
        rate = pd.to_numeric(eval_df.get("disagreement_item_rate", pd.Series(np.nan, index=eval_df.index)), errors="coerce")
        exclude_by_rate = rate >= disagree_rate_exclude_thresh
        exclude_by_row_level = eval_df.get("row_level_variant_disagreement", pd.Series(False, index=eval_df.index)).fillna(False).astype(bool)
        exclude_mask = exclude_by_rate | exclude_by_row_level
        eval_df = eval_df[~exclude_mask].copy()
        print(f"Excluded rows with disagreement_item_rate >= {disagree_rate_exclude_thresh}: {int(exclude_by_rate.sum())} / {before_eval}")
        print(f"Excluded rows with row-level variant disagreement (safety): {int(exclude_by_row_level.sum())} / {before_eval}")
        print(f"Total excluded rows (union): {before_eval - len(eval_df)} / {before_eval}")

    print("Evaluated speakers (rows in eval_df, post-filter):", len(eval_df))
    if len(eval_df) > 0 and "speaker" in eval_df.columns:
        print("Evaluated speaker breakdown:\n", eval_df["speaker"].value_counts(dropna=False))
    else:
        print("Evaluated speaker breakdown:\n", "(no evaluated speakers)")

    # How many conversations have A-only / B-only / both?
    has_a = df_wide["avg_score_a"].apply(lambda x: str(x).strip() not in ("", "nan", "NaN")) if "avg_score_a" in df_wide.columns else pd.Series(False, index=df_wide.index)
    has_b = df_wide["avg_score_b"].apply(lambda x: str(x).strip() not in ("", "nan", "NaN")) if "avg_score_b" in df_wide.columns else pd.Series(False, index=df_wide.index)
    print("\nConversation-level coverage:")
    print("A-only:", int((has_a & ~has_b).sum()))
    print("B-only:", int((~has_a & has_b).sum()))
    print("Both:", int((has_a & has_b).sum()))
    print("Neither:", int((~has_a & ~has_b).sum()))

    MODEL_EVAL_DFS[model_name] = eval_df

print("\nLoaded models:", MODEL_ORDER)


In [ ]:
if POS_CATEGORY is not None and NEG_CATEGORY is not None:
    pos_set = set(POS_CATEGORY) if isinstance(POS_CATEGORY, (list, tuple, set)) else ({POS_CATEGORY} if POS_CATEGORY is not None else set())
    neg_set = set(NEG_CATEGORY) if isinstance(NEG_CATEGORY, (list, tuple, set)) else ({NEG_CATEGORY} if NEG_CATEGORY is not None else set())
    def expected_cat_pole(cat):
        if cat in pos_set:
            return 1
        if cat in neg_set:
            return -1
        return 0

    for model_name in MODEL_ORDER:
        eval_df = MODEL_EVAL_DFS[model_name]

        print("\n" + "-" * 90)
        print(f"MODEL: {model_name}")

        if len(eval_df) == 0 or "category" not in eval_df.columns:
            print("[WARN] No evaluated rows with category; skipping categorical consistency.")
            continue

        eval_df["exp_cat_pole"] = eval_df["category"].apply(expected_cat_pole)

        # treat score_used==0 as "uncertain" and exclude from pole-consistency
        mask = (eval_df["exp_cat_pole"] != 0) & (eval_df["pred_pole"] != 0)
        cat_acc = (eval_df.loc[mask, "pred_pole"] == eval_df.loc[mask, "exp_cat_pole"]).mean()

        print("Categorical pole-consistency (excluding score==0):", cat_acc)

        # Equal Error Rate (EER) on the same filtered subset
        eer = np.nan
        y_true_cat = (eval_df.loc[mask, "exp_cat_pole"] == 1).astype(int).values
        y_score_cat = eval_df.loc[mask, "score_used"].astype(float).values
        if len(y_true_cat) > 0 and len(np.unique(y_true_cat)) == 2:
            try:
                from sklearn.metrics import roc_curve
                fpr_cat, tpr_cat, _ = roc_curve(y_true_cat, y_score_cat)
                fnr_cat = 1.0 - tpr_cat
                idx_cat = int(np.nanargmin(np.abs(fnr_cat - fpr_cat)))
                eer = float((fpr_cat[idx_cat] + fnr_cat[idx_cat]) / 2.0)
            except Exception:
                thr_cat = np.sort(np.unique(y_score_cat))
                fpr_cat, fnr_cat = [], []
                for t in thr_cat:
                    pred = (y_score_cat >= t).astype(int)
                    tp = ((pred == 1) & (y_true_cat == 1)).sum()
                    fp = ((pred == 1) & (y_true_cat == 0)).sum()
                    fn = ((pred == 0) & (y_true_cat == 1)).sum()
                    tn = ((pred == 0) & (y_true_cat == 0)).sum()
                    fpr_cat.append(fp / (fp + tn) if (fp + tn) else 0.0)
                    fnr_cat.append(fn / (fn + tp) if (fn + tp) else 0.0)
                fpr_cat = np.array(fpr_cat)
                fnr_cat = np.array(fnr_cat)
                idx_cat = int(np.nanargmin(np.abs(fnr_cat - fpr_cat)))
                eer = float((fpr_cat[idx_cat] + fnr_cat[idx_cat]) / 2.0)
        print("Equal Error Rate (EER):", eer)

        eligible_cat = (eval_df["exp_cat_pole"] != 0)
        uncertain_rate = (
            ((eligible_cat) & (eval_df["pred_pole"] == 0)).sum() / eligible_cat.sum()
            if eligible_cat.sum() else np.nan
        )
        print("Uncertain rate:", uncertain_rate)

        # Oracle upper bound: tune (tau0, tau2) and a decision threshold
        # (threshold scale depends on use_prob: prob when True, score when False).
        # using category-labeled rows only (exp_cat_pole in {-1, +1}).
        def _coerce_prob_item(item, vidx):
            p = safe_float(item.get(f"prob_v{vidx}", np.nan))
            if not np.isfinite(p):
                p = safe_float(item.get(f"probablity_v{vidx}", np.nan))
            if np.isfinite(p) and p > 1.0:
                p = p / 100.0
            if np.isfinite(p):
                p = float(np.clip(p, 0.0, 1.0))
            return p

        def _map_choice_prob(choice, prob, tau0, tau2):
            c = str(choice).strip().upper()
            if c not in {"P", "N"} or not np.isfinite(prob):
                return None
            s = 1 if c == "P" else -1
            if prob < float(tau0):
                return 0
            if prob >= float(tau2):
                return 2 * s
            return 1 * s

        def _row_score_with_taus(turns, tau0, tau2):
            vals = []
            if not isinstance(turns, list):
                return np.nan
            for t in turns:
                c0 = t.get("choice_v0", None)
                c1 = t.get("choice_v1", None)
                p0 = _coerce_prob_item(t, 0)
                p1 = _coerce_prob_item(t, 1)
                s0 = _map_choice_prob(c0, p0, tau0, tau2)
                s1 = _map_choice_prob(c1, p1, tau0, tau2)
                if s0 is None or s1 is None:
                    continue
                vals.append((float(s0) + float(s1)) / 2.0)
            return float(np.mean(vals)) if vals else np.nan

        def _decision_signal_from_score(arr_like):
            arr = np.asarray(arr_like, dtype=float)
            if use_prob:
                # Map score scale [-2, 2] to probability-like scale [0, 1].
                return np.clip((arr + 2.0) / 4.0, 0.0, 1.0)
            return arr

        decision_threshold_scale = "prob" if use_prob else "score"

        oracle_rows = eval_df.loc[eligible_cat, ["exp_cat_pole", "evidence_turnovers"]].copy()
        y_true_oracle = oracle_rows["exp_cat_pole"].astype(int).values

        tau_grid = np.round(np.arange(0.0, 1.0001, 0.05), 2)
        best_oracle = {
            "f1": -1.0,
            "tau0": np.nan,
            "tau2": np.nan,
            "thr": np.nan,
            "n_used": 0,
        }
        best_oracle_degenerate = {
            "f1": -1.0,
            "tau0": np.nan,
            "tau2": np.nan,
            "thr": np.nan,
            "n_used": 0,
        }

        for tau0 in tau_grid:
            for tau2 in tau_grid:
                if tau2 < tau0:
                    continue

                score_tau = oracle_rows["evidence_turnovers"].apply(
                    lambda turns: _row_score_with_taus(turns, tau0, tau2)
                ).astype(float).values
                valid = np.isfinite(score_tau)
                if valid.sum() == 0:
                    continue

                s = _decision_signal_from_score(score_tau[valid])
                y = y_true_oracle[valid]

                uniq = np.unique(s)
                if len(uniq) <= 1:
                    continue
                # Use only interior thresholds (between observed scores) to avoid
                # all-positive / all-negative edge-threshold solutions.
                cand_thr = ((uniq[:-1] + uniq[1:]) / 2.0).astype(float).tolist()

                for thr in cand_thr:
                    # Single decision threshold on the active scale (prob or score).
                    pred = np.where(s >= float(thr), 1, -1)
                    tp = int(((pred == 1) & (y == 1)).sum())
                    fp = int(((pred == 1) & (y == -1)).sum())
                    fn = int(((pred == -1) & (y == 1)).sum())
                    denom = (2 * tp) + fp + fn
                    f1 = float((2 * tp) / denom) if denom else 0.0
                    n_used = int(len(y))
                    is_degenerate = (np.unique(pred).size < 2)
                    if is_degenerate:
                        if (f1 > best_oracle_degenerate["f1"] + 1e-12) or (
                            abs(f1 - best_oracle_degenerate["f1"]) <= 1e-12 and n_used > best_oracle_degenerate["n_used"]
                        ):
                            best_oracle_degenerate["f1"] = f1
                            best_oracle_degenerate["tau0"] = float(tau0)
                            best_oracle_degenerate["tau2"] = float(tau2)
                            best_oracle_degenerate["thr"] = float(thr)
                            best_oracle_degenerate["n_used"] = n_used
                        continue

                    if (f1 > best_oracle["f1"] + 1e-12) or (
                        abs(f1 - best_oracle["f1"]) <= 1e-12 and n_used > best_oracle["n_used"]
                    ):
                        best_oracle["f1"] = f1
                        best_oracle["tau0"] = float(tau0)
                        best_oracle["tau2"] = float(tau2)
                        best_oracle["thr"] = float(thr)
                        best_oracle["n_used"] = n_used

        if best_oracle["f1"] >= 0:
            score_best = oracle_rows["evidence_turnovers"].apply(
                lambda turns: _row_score_with_taus(turns, best_oracle["tau0"], best_oracle["tau2"])
            ).astype(float).values
            valid_best = np.isfinite(score_best)
            s_best = _decision_signal_from_score(score_best[valid_best])
            pred_best = np.where(s_best >= float(best_oracle["thr"]), 1, -1)
            pred_pos = int((pred_best == 1).sum())
            pred_neg = int((pred_best == -1).sum())
            print("Oracle upper bound f1:", best_oracle["f1"])
            print("Oracle decision-threshold scale:", decision_threshold_scale)
            print(
                "Oracle best params:",
                {
                    "tau0": best_oracle["tau0"],
                    "tau2": best_oracle["tau2"],
                    "decision_threshold": best_oracle["thr"],
                    "decision_threshold_scale": decision_threshold_scale,
                },
            )
            print("Oracle pred counts:", {"pos": pred_pos, "neg": pred_neg})
            print("Oracle N used:", best_oracle["n_used"])
        elif best_oracle_degenerate["f1"] >= 0:
            print("[WARN] Oracle search found only degenerate (single-class) thresholds; no oracle params reported.")
            print(
                "[WARN] Best degenerate params:",
                {
                    "tau0": best_oracle_degenerate["tau0"],
                    "tau2": best_oracle_degenerate["tau2"],
                    "decision_threshold": best_oracle_degenerate["thr"],
                    "decision_threshold_scale": decision_threshold_scale,
                    "f1": best_oracle_degenerate["f1"],
                },
            )
        else:
            print("[WARN] Oracle search skipped: no valid rows with usable evidence_turnovers.")

        print("N used:", int(mask.sum()))
        item_flip_rate_mean = MODEL_ITEM_FLIP_RATE_MEAN_PRE.get(model_name, np.nan)
        print("Mean item disagreement rate:", item_flip_rate_mean)

        # Confusion-like table (pos/neg only)
        tmp = eval_df.loc[mask, ["exp_cat_pole", "pred_pole"]].copy()
        tmp["exp"] = tmp["exp_cat_pole"].map({1: "POS_CAT", -1: "NEG_CAT"})
        tmp["pred"] = tmp["pred_pole"].map({1: "POS", -1: "NEG"})
        ct = pd.crosstab(tmp["exp"], tmp["pred"])
        display(ct)

else:
    print("Skipping categorical consistency (POS_CATEGORY/NEG_CATEGORY not set).")


In [ ]:
# --- Build merged two-group arrays and violin-style plots (per model) ---

def p_to_stars(p):
    if not np.isfinite(p):
        return None
    if p < 0.001:
        return "***"
    if p < 0.01:
        return "**"
    if p < 0.05:
        return "*"
    return None


def add_sig_bracket(ax, x1, x2, y, h, text):
    ax.plot([x1, x1, x2, x2],
            [y, y + h, y + h, y],
            linewidth=1,
            color="black")
    ax.text((x1 + x2) / 2,
            y + h,
            text,
            ha="center",
            va="bottom",
            color="black")

for model_name in MODEL_ORDER:
    eval_df = MODEL_EVAL_DFS[model_name]
    plot_label = MODEL_PLOT_LABEL.get(model_name, "score")
    plot_refs = MODEL_PLOT_REF_LEVELS.get(model_name, [-2, -1, 0, 1, 2])

    pos_scores = eval_df.loc[
        eval_df["role"].isin(POS_ROLES), "score_plot"
    ].astype(float).dropna().values

    neg_scores = eval_df.loc[
        eval_df["role"].isin(NEG_ROLES), "score_plot"
    ].astype(float).dropna().values

    print("\n" + "-" * 90)
    print(f"MODEL: {model_name}")

    if len(pos_scores) == 0 or len(neg_scores) == 0:
        print("[WARN] One of the groups is empty; cannot plot POS vs NEG violin plot.")
        continue

    # Mann–Whitney significance
    u = mannwhitneyu(pos_scores, neg_scores, alternative="two-sided")
    pval = u.pvalue
    stars = p_to_stars(pval)

    # --- Plot ---
    fig, ax = plt.subplots(figsize=(5, 4))
    from scipy.stats import gaussian_kde
    groups = [pos_scores, neg_scores]
    y_lo = min(float(np.min(v)) for v in groups if len(v) > 0)
    y_hi = max(float(np.max(v)) for v in groups if len(v) > 0)
    if not np.isfinite(y_lo) or not np.isfinite(y_hi) or y_lo == y_hi:
        y_lo, y_hi = y_lo - 0.5, y_hi + 0.5
    y_grid = np.linspace(y_lo, y_hi, 256)

    dens_list = []
    for vals in groups:
        vals = np.asarray(vals, dtype=float)
        vals = vals[np.isfinite(vals)]
        if len(vals) >= 2 and np.std(vals) > 0:
            d = gaussian_kde(vals)(y_grid)
        elif len(vals) == 1:
            bw = max((y_hi - y_lo) / 40.0, 1e-6)
            d = np.exp(-0.5 * ((y_grid - vals[0]) / bw) ** 2)
        else:
            d = np.zeros_like(y_grid)
        if len(vals) > 0:
            vmin, vmax = float(np.min(vals)), float(np.max(vals))
            d[(y_grid < vmin) | (y_grid > vmax)] = 0.0
        dens_list.append(d)

    global_max = max(float(np.max(d)) for d in dens_list) if dens_list else 0.0
    if not np.isfinite(global_max) or global_max <= 0:
        global_max = 1.0

    max_half_width = 0.35
    for i, (vals, dens) in enumerate(zip(groups, dens_list), start=1):
        w = (dens / global_max) * max_half_width
        ax.fill_betweenx(y_grid, i - w, i + w, color="C0", alpha=0.35, linewidth=0)
        vals = np.asarray(vals, dtype=float)
        vals = vals[np.isfinite(vals)]
        if len(vals) > 0:
            vmin, vmax = float(np.min(vals)), float(np.max(vals))
            med = float(np.median(vals))
            minmax_half = 0.08
            median_half = 0.14
            ax.plot([i, i], [vmin, vmax], color="grey", linewidth=1)
            ax.plot([i - minmax_half, i + minmax_half], [vmin, vmin], color="grey", linewidth=1.2)
            ax.plot([i - minmax_half, i + minmax_half], [vmax, vmax], color="grey", linewidth=1.2)
            ax.plot([i - median_half, i + median_half], [med, med], color="orange", linewidth=1.5)

    rng = np.random.default_rng(42)
    jitter = 0.04
    for i, vals in enumerate([pos_scores, neg_scores], start=1):
        x = i + rng.uniform(-jitter, jitter, size=len(vals))
        ax.scatter(x, vals, s=12, alpha=0.35, color="black", linewidths=0, zorder=1)

    ax.set_xlim(0.5, 2.5)
    ax.set_xticks([1, 2], labels=[f"POS ({POS_CATEGORY_LABEL})", f"NEG ({NEG_CATEGORY_LABEL})"])

    # Horizontal dashed reference lines (scaled by score mode)
    for y_ref in plot_refs:
        ax.axhline(y_ref, linestyle="--", linewidth=1)

    ax.set_title(f"{model_name}: Merged groups: POS vs NEG categories ({plot_label})")
    ax.set_ylabel(plot_label)

    # Significance bracket (only if significant)
    if stars is not None:
        y_max = max(np.max(pos_scores), np.max(neg_scores))
        y_min = min(np.min(pos_scores), np.min(neg_scores))
        span = (y_max - y_min) if y_max > y_min else 1.0

        h = 0.08 * span
        y = y_max + 0.12 * span

        # bracket between box 1 and box 2 (black, hard-coded)
        add_sig_bracket(ax, 1, 2, y, h, stars)
        ax.set_ylim(top=y + h + 0.2 * span)

    plt.tight_layout()
    plt.show()

    if np.isfinite(pval):
        print(f"{model_name} | Mann–Whitney p:", pval)


In [ ]:
# --- ROC/AUC for POS vs NEG discrimination using score_used (all models in one plot) ---

def compute_roc_auc(y_true, y_score):
    """Return (fpr, tpr, auc_val) using sklearn when available, else a fallback."""
    if len(np.unique(y_true)) < 2:
        return None

    try:
        from sklearn.metrics import roc_curve, roc_auc_score
        fpr, tpr, _ = roc_curve(y_true, y_score)
        auc_val = roc_auc_score(y_true, y_score)
        return (fpr, tpr, float(auc_val))
    except Exception:
        # fallback: AUC from Mann–Whitney (probability a random POS has higher score than random NEG)
        pos = y_score[y_true == 1]
        neg = y_score[y_true == 0]
        u = mannwhitneyu(pos, neg, alternative="two-sided")
        auc_val = float(u.statistic / (len(pos) * len(neg)))

        # crude ROC curve fallback (manual thresholds)
        thr = np.unique(y_score)
        thr = np.sort(thr)
        fpr, tpr = [], []
        for t in thr:
            pred = (y_score >= t).astype(int)
            tp = ((pred == 1) & (y_true == 1)).sum()
            fp = ((pred == 1) & (y_true == 0)).sum()
            fn = ((pred == 0) & (y_true == 1)).sum()
            tn = ((pred == 0) & (y_true == 0)).sum()
            tpr.append(tp / (tp + fn) if (tp + fn) else 0.0)
            fpr.append(fp / (fp + tn) if (fp + tn) else 0.0)
        fpr = np.array(fpr)
        tpr = np.array(tpr)
        return (fpr, tpr, auc_val)

if len(MODEL_ORDER) == 0:
    print("[WARN] No models loaded; skipping ROC/AUC.")
else:
    fig, ax = plt.subplots(figsize=(6, 5))

    any_curve = False

    for model_name in MODEL_ORDER:
        eval_df = MODEL_EVAL_DFS[model_name]
        score_label = MODEL_SCORE_LABEL.get(model_name, "score_used")

        merged = eval_df[eval_df["role"].isin(set(POS_ROLES) | set(NEG_ROLES))].copy()
        merged = merged.dropna(subset=["score_used"])
        merged["y_true"] = merged["role"].isin(POS_ROLES).astype(int)   # POS=1, NEG=0

        y_true = merged["y_true"].values
        y_score = merged["score_used"].astype(float).values

        res = compute_roc_auc(y_true, y_score)
        if res is None:
            print(f"[WARN] {model_name}: Need both POS and NEG samples to compute ROC/AUC.")
            continue

        fpr, tpr, auc_val = res
        any_curve = True

        (line,) = ax.plot(fpr, tpr, linewidth=2, label=f"{model_name} ({score_label}, AUC={auc_val:.3f})")
        ax.fill_between(fpr, tpr, 0, alpha=0.15, color=line.get_color())

    ax.plot([0, 1], [0, 1], linestyle="--", linewidth=1, color="gray")
    ax.set_title("ROC: POS vs NEG roles")
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    if any_curve:
        ax.legend(loc="lower right")
    plt.tight_layout()
    plt.show()


In [ ]:
def _roles_to_list(x):
    if x is None:
        return []
    if isinstance(x, (list, tuple, set)):
        return [str(r).strip() for r in x if str(r).strip()]
    s = str(x).strip()
    if not s:
        return []
    parts = re.split(r"[,\s]+", s)
    return [p.strip() for p in parts if p.strip()]

# POS_ROLES / NEG_ROLES are defined in an earlier cell; support list or string
POS_ROLES_list = _roles_to_list(POS_ROLES)
NEG_ROLES_list = _roles_to_list(NEG_ROLES)

POS_ROLES_set = set(POS_ROLES_list)
NEG_ROLES_set = set(NEG_ROLES_list)

def expected_role_pole(role):
    if role in POS_ROLES_set:
        return 1
    if role in NEG_ROLES_set:
        return -1
    return 0

for model_name in MODEL_ORDER:
    eval_df = MODEL_EVAL_DFS[model_name]
    plot_label = MODEL_PLOT_LABEL.get(model_name, "score")
    plot_refs = MODEL_PLOT_REF_LEVELS.get(model_name, [-2, -1, 0, 1, 2])

    print("\n" + "-" * 90)
    print(f"MODEL: {model_name}")

    if len(eval_df) == 0:
        print("[WARN] No evaluated rows; skipping per-role plotting.")
        continue

    eval_df["exp_role_pole"] = eval_df["role"].apply(expected_role_pole)

    # Accuracy-style: exclude unknown roles and exclude score==0
    mask = (eval_df["exp_role_pole"] != 0) & (eval_df["pred_pole"] != 0)
    role_acc = (eval_df.loc[mask, "pred_pole"] == eval_df.loc[mask, "exp_role_pole"]).mean() if int(mask.sum()) else np.nan
    print("Role pole-consistency (excluding score==0):", role_acc)
    print("N used:", int(mask.sum()))

    # --- Violin plot grouped by POS roles then NEG roles (preserve user-specified order) ---
    # Keep only roles that actually appear
    present_roles = set(eval_df["role"].astype(str).unique().tolist())
    pos_roles_order = [r for r in POS_ROLES_list if r in present_roles]
    neg_roles_order = [r for r in NEG_ROLES_list if r in present_roles]
    roles_plot_order = pos_roles_order + neg_roles_order

    if len(roles_plot_order) == 0:
        print("[WARN] No roles present after filtering; skipping per-role plot.")
        continue

    plot_df = eval_df[eval_df["role"].isin(roles_plot_order)].copy()

    data = [
        plot_df.loc[plot_df["role"] == r, "score_plot"].astype(float).dropna().values
        for r in roles_plot_order
    ]

    fig, ax = plt.subplots(figsize=(max(10, 0.8 * len(roles_plot_order)), 5))
    from scipy.stats import gaussian_kde
    flat = np.concatenate([np.asarray(v, dtype=float) for v in data if len(v) > 0]) if any(len(v) > 0 for v in data) else np.array([0.0, 1.0])
    flat = flat[np.isfinite(flat)]
    if flat.size == 0:
        flat = np.array([0.0, 1.0])
    x_lo, x_hi = float(np.min(flat)), float(np.max(flat))
    if not np.isfinite(x_lo) or not np.isfinite(x_hi) or x_lo == x_hi:
        x_lo, x_hi = x_lo - 0.5, x_hi + 0.5
    x_grid = np.linspace(x_lo, x_hi, 256)

    dens_list = []
    for vals in data:
        vals = np.asarray(vals, dtype=float)
        vals = vals[np.isfinite(vals)]
        if len(vals) >= 2 and np.std(vals) > 0:
            d = gaussian_kde(vals)(x_grid)
        elif len(vals) == 1:
            bw = max((x_hi - x_lo) / 40.0, 1e-6)
            d = np.exp(-0.5 * ((x_grid - vals[0]) / bw) ** 2)
        else:
            d = np.zeros_like(x_grid)
        if len(vals) > 0:
            vmin, vmax = float(np.min(vals)), float(np.max(vals))
            d[(x_grid < vmin) | (x_grid > vmax)] = 0.0
        dens_list.append(d)

    global_max = max(float(np.max(d)) for d in dens_list) if dens_list else 0.0
    if not np.isfinite(global_max) or global_max <= 0:
        global_max = 1.0

    # Robust global reference (not per-role normalization).
    group_maxes = np.array([float(np.max(d)) for d in dens_list]) if dens_list else np.array([global_max])
    nz = group_maxes[group_maxes > 0]
    if nz.size > 0:
        density_ref = float(np.nanpercentile(nz, 85))
        if not np.isfinite(density_ref) or density_ref <= 0:
            density_ref = global_max
    else:
        density_ref = global_max

    # Build absolute widths, then space rows dynamically to avoid overlap.
    width_scale = 0.35
    min_visible_half_width = 0.015
    gamma = 0.6
    width_list = []
    for dens in dens_list:
        dens_norm = dens / density_ref
        w = (dens_norm ** gamma) * width_scale
        w = np.where(dens > 0, np.maximum(w, min_visible_half_width), 0.0)
        width_list.append(w)

    half_max = np.array([float(np.max(w)) if w.size else 0.0 for w in width_list]) if width_list else np.array([])
    centers = np.array([], dtype=float)
    if len(data) > 0:
        centers = np.zeros(len(data), dtype=float)
        centers[0] = 1.0
        gap = 0.25
        min_step = 0.9
        for j in range(1, len(data)):
            step = float(half_max[j - 1] + half_max[j] + gap)
            centers[j] = centers[j - 1] + max(step, min_step)

    for c, vals, w in zip(centers, data, width_list):
        ax.fill_between(x_grid, c - w, c + w, color="C0", alpha=0.35, linewidth=0)
        vals = np.asarray(vals, dtype=float)
        vals = vals[np.isfinite(vals)]
        if len(vals) > 0:
            vmin, vmax = float(np.min(vals)), float(np.max(vals))
            med = float(np.median(vals))
            minmax_half = 0.16
            median_half = 0.28
            ax.plot([vmin, vmax], [c, c], color="grey", linewidth=1)
            ax.plot([vmin, vmin], [c - minmax_half, c + minmax_half], color="grey", linewidth=1)
            ax.plot([vmax, vmax], [c - minmax_half, c + minmax_half], color="grey", linewidth=1)
            ax.plot([med, med], [c - median_half, c + median_half], color="orange", linewidth=1.5)

    rng = np.random.default_rng(42)
    jitter = 0.08
    for c, vals in zip(centers, data):
        y = c + rng.uniform(-jitter, jitter, size=len(vals))
        ax.scatter(vals, y, s=10, alpha=0.3, color="black", linewidths=0, zorder=1)

    ax.set_yticks(centers, labels=roles_plot_order)
    if centers.size > 0:
        pad = float(np.max(half_max) + 0.25) if half_max.size > 0 else 0.5
        ax.set_ylim(float(centers[0] - pad), float(centers[-1] + pad))

    # Vertical dashed reference lines (scaled by score mode)
    for x in plot_refs:
        ax.axvline(x, linestyle="--", linewidth=1, color='grey')

    # Horizontal separator between POS and NEG groups (if both exist)
    if len(pos_roles_order) > 0 and len(neg_roles_order) > 0:
        i_pos_last = len(pos_roles_order) - 1
        i_neg_first = len(pos_roles_order)
        sep_y = (float(centers[i_pos_last]) + float(centers[i_neg_first])) / 2.0
        ax.axhline(sep_y, linestyle=":", linewidth=1, color='black')

    ax.set_title(f"{model_name}: Score distribution by role (POS group then NEG group): {plot_label}")
    ax.set_xlabel(plot_label)
    plt.tight_layout()
    plt.show()

    # --- Separation metrics (POS roles vs NEG roles) ---
    pos_scores = eval_df.loc[eval_df["role"].isin(POS_ROLES_set), "score_used"].astype(float).dropna().values
    neg_scores = eval_df.loc[eval_df["role"].isin(NEG_ROLES_set), "score_used"].astype(float).dropna().values

    if len(pos_scores) > 0:
        print("POS roles mean:", float(np.mean(pos_scores)), "N:", len(pos_scores))
    else:
        print("POS roles mean: (no samples)")

    if len(neg_scores) > 0:
        print("NEG roles mean:", float(np.mean(neg_scores)), "N:", len(neg_scores))
    else:
        print("NEG roles mean: (no samples)")

    def cohens_d(x, y):
        x = np.array(x); y = np.array(y)
        nx, ny = len(x), len(y)
        if nx < 2 or ny < 2:
            return np.nan
        sx, sy = x.std(ddof=1), y.std(ddof=1)
        sp = math.sqrt(((nx-1)*sx*sx + (ny-1)*sy*sy) / (nx+ny-2))
        return (x.mean() - y.mean()) / sp if sp > 0 else np.nan

    print("Cohen's d (POS - NEG):", cohens_d(pos_scores, neg_scores))

    if len(pos_scores) > 0 and len(neg_scores) > 0:
        u = mannwhitneyu(pos_scores, neg_scores, alternative="two-sided")
        auc = u.statistic / (len(pos_scores) * len(neg_scores))
        print("Mann–Whitney p:", u.pvalue)
        print("AUC (P[pos>neg]):", float(auc))
    else:
        print("scipy not available (or insufficient samples); skipping Mann–Whitney/AUC.")


In [ ]:
# Multi-question summary runner
IDX_LIST = [0,1,2,3,4,5,6,7,8]
absolute_category_violin = True  # True -> category violin thickness is absolute across all models
prob_decision_threshold_summary = globals().get("prob_decision_threshold", 0.0)

def _coerce_prob_item_summary(item, vidx):
    p = safe_float(item.get(f"prob_v{vidx}", np.nan))
    if not np.isfinite(p):
        p = safe_float(item.get(f"probablity_v{vidx}", np.nan))
    if np.isfinite(p) and p > 1.0:
        p = p / 100.0
    if np.isfinite(p):
        p = float(np.clip(p, 0.0, 1.0))
    return p

def _item_signed_prob_summary(choice, prob):
    c = str(choice).strip().upper()
    if c not in {"P", "N"} or not np.isfinite(prob):
        return np.nan
    return float(prob) if c == "P" else float(-prob)

def _row_signed_prob_direct_summary(turns):
    vals = []
    if not isinstance(turns, list):
        return np.nan
    for t in turns:
        p0 = _coerce_prob_item_summary(t, 0)
        p1 = _coerce_prob_item_summary(t, 1)
        q0 = _item_signed_prob_summary(t.get("choice_v0", None), p0)
        q1 = _item_signed_prob_summary(t.get("choice_v1", None), p1)
        if not np.isfinite(q0) or not np.isfinite(q1):
            continue
        vals.append((float(q0) + float(q1)) / 2.0)
    return float(np.mean(vals)) if vals else np.nan

def _roles_for_idx(idx):
    role_map = {
        0: ("Friendly Warm Approachable Welcoming", "Aloof Distant Impersonal Indifferent"),
        1: ("Empathetic Considerate Understanding Concerned", "Insensitive Unsympathetic Inconsiderate Callous"),
        2: ("Polite Respectful Courteous", "Disrespectful Impolite Uncivil"),
        3: ("Assertive Decisive Self-assured Firm", "Indecisive Self-doubting Unassertive Timid"),
        4: ("Honest Ingenuous Uncalculating", "Manipulative Calculating Devious"),
        5: ("Alert Attentive Concentrating Engaged", "Bewildered Distracted Drowsy Unfocused"),
        6: ("Engaging Sociable Gregarious Outgoing", "Withdrawn Disengaged Reticent Taciturn"),
        7: ("Submissive Meek Yielding Undemanding", "Forceful Overbearing Dominant Domineering"),
        8: ("Stable Steady Unaggressive Unargumentative", "Aggressive Cruel Ruthless Vindictive"),
    }
    if idx not in role_map:
        raise ValueError(f"No POS/NEG roles configured for IDX={idx}")
    pos, neg = role_map[idx]
    return parse_roles(pos), parse_roles(neg)


def _resolve_question(project_root, run_root_name, idx, subfolders):
    for subfolder, model_name in subfolders.items():
        p = resolve_question_path(project_root, run_root_name, subfolder, model_name, idx)
        if p.exists():
            return load_question(p)
    return load_bundled_question(idx)


def _compute_perf_metrics(eval_df, pos_category, neg_category, use_prob_flag):
    out = {
        "Categorical pole-consistency": np.nan,
        "Equal Error Rate": np.nan,
        "AUROC": np.nan,
        "Oracle upper bound f1": np.nan,
    }
    if len(eval_df) == 0 or ("category" not in eval_df.columns):
        return out

    pos_set = set(pos_category) if isinstance(pos_category, (list, tuple, set)) else ({pos_category} if pos_category is not None else set())
    neg_set = set(neg_category) if isinstance(neg_category, (list, tuple, set)) else ({neg_category} if neg_category is not None else set())

    def expected_cat_pole(cat):
        if cat in pos_set:
            return 1
        if cat in neg_set:
            return -1
        return 0

    dd = eval_df.copy()
    dd["exp_cat_pole"] = dd["category"].apply(expected_cat_pole)
    mask = (dd["exp_cat_pole"] != 0) & (dd["pred_pole"] != 0)
    out["Categorical pole-consistency"] = (dd.loc[mask, "pred_pole"] == dd.loc[mask, "exp_cat_pole"]).mean()

    y_true_cat = (dd.loc[mask, "exp_cat_pole"] == 1).astype(int).values
    y_score_cat = dd.loc[mask, "score_used"].astype(float).values
    if len(y_true_cat) > 0 and len(np.unique(y_true_cat)) == 2:
        try:
            from sklearn.metrics import roc_curve, roc_auc_score
            fpr_cat, tpr_cat, _ = roc_curve(y_true_cat, y_score_cat)
            fnr_cat = 1.0 - tpr_cat
            idx_cat = int(np.nanargmin(np.abs(fnr_cat - fpr_cat)))
            out["Equal Error Rate"] = float((fpr_cat[idx_cat] + fnr_cat[idx_cat]) / 2.0)
            out["AUROC"] = float(roc_auc_score(y_true_cat, y_score_cat))
        except Exception:
            thr_cat = np.sort(np.unique(y_score_cat))
            fpr_cat, fnr_cat, tpr_cat = [], [], []
            for t in thr_cat:
                pred = (y_score_cat >= t).astype(int)
                tp = ((pred == 1) & (y_true_cat == 1)).sum()
                fp = ((pred == 1) & (y_true_cat == 0)).sum()
                fn = ((pred == 0) & (y_true_cat == 1)).sum()
                tn = ((pred == 0) & (y_true_cat == 0)).sum()
                fpr_cat.append(fp / (fp + tn) if (fp + tn) else 0.0)
                fnr_cat.append(fn / (fn + tp) if (fn + tp) else 0.0)
                tpr_cat.append(tp / (tp + fn) if (tp + fn) else 0.0)
            fpr_cat = np.array(fpr_cat)
            fnr_cat = np.array(fnr_cat)
            tpr_cat = np.array(tpr_cat)
            idx_cat = int(np.nanargmin(np.abs(fnr_cat - fpr_cat)))
            out["Equal Error Rate"] = float((fpr_cat[idx_cat] + fnr_cat[idx_cat]) / 2.0)
            if len(fpr_cat) > 1:
                order = np.argsort(fpr_cat)
                out["AUROC"] = float(np.trapz(tpr_cat[order], fpr_cat[order]))

    eligible_cat = (dd["exp_cat_pole"] != 0)

    def _coerce_prob_item(item, vidx):
        return _coerce_prob_item_summary(item, vidx)

    def _item_signed_prob(choice, prob):
        return _item_signed_prob_summary(choice, prob)

    def _row_prob_direct(turns):
        vals = []
        if not isinstance(turns, list):
            return np.nan
        for t in turns:
            c0 = t.get("choice_v0", None)
            c1 = t.get("choice_v1", None)
            p0 = _coerce_prob_item(t, 0)
            p1 = _coerce_prob_item(t, 1)
            q0 = _item_signed_prob(c0, p0)
            q1 = _item_signed_prob(c1, p1)
            if not np.isfinite(q0) or not np.isfinite(q1):
                continue
            vals.append((float(q0) + float(q1)) / 2.0)
        return float(np.mean(vals)) if vals else np.nan

    def _map_choice_prob(choice, prob, tau0, tau2):
        c = str(choice).strip().upper()
        if c not in {"P", "N"} or not np.isfinite(prob):
            return None
        s = 1 if c == "P" else -1
        if prob < float(tau0):
            return 0
        if prob >= float(tau2):
            return 2 * s
        return 1 * s

    def _row_score_with_taus(turns, tau0, tau2):
        vals = []
        if not isinstance(turns, list):
            return np.nan
        for t in turns:
            c0 = t.get("choice_v0", None)
            c1 = t.get("choice_v1", None)
            p0 = _coerce_prob_item(t, 0)
            p1 = _coerce_prob_item(t, 1)
            s0 = _map_choice_prob(c0, p0, tau0, tau2)
            s1 = _map_choice_prob(c1, p1, tau0, tau2)
            if s0 is None or s1 is None:
                continue
            vals.append((float(s0) + float(s1)) / 2.0)
        return float(np.mean(vals)) if vals else np.nan

    oracle_rows = dd.loc[eligible_cat, ["exp_cat_pole", "evidence_turnovers"]].copy()
    y_true_oracle = oracle_rows["exp_cat_pole"].astype(int).values

    if use_prob_flag:
        best_oracle = {"f1": -1.0, "thr": np.nan, "n_used": 0}
        prob_signal = oracle_rows["evidence_turnovers"].apply(_row_prob_direct).astype(float).values
        valid = np.isfinite(prob_signal)
        if valid.sum() > 0:
            s = prob_signal[valid]
            y = y_true_oracle[valid]
            uniq = np.unique(s)
            if len(uniq) > 1:
                cand_thr = ((uniq[:-1] + uniq[1:]) / 2.0).astype(float).tolist()
                for thr in cand_thr:
                    pred = np.where(s >= float(thr), 1, -1)
                    if np.unique(pred).size < 2:
                        continue
                    tp = int(((pred == 1) & (y == 1)).sum())
                    fp = int(((pred == 1) & (y == -1)).sum())
                    fn = int(((pred == -1) & (y == 1)).sum())
                    denom = (2 * tp) + fp + fn
                    f1 = float((2 * tp) / denom) if denom else 0.0
                    n_used = int(len(y))
                    if (f1 > best_oracle["f1"] + 1e-12) or (
                        abs(f1 - best_oracle["f1"]) <= 1e-12 and n_used > best_oracle["n_used"]
                    ):
                        best_oracle = {"f1": f1, "thr": float(thr), "n_used": n_used}
    else:
        tau_grid = np.round(np.arange(0.0, 1.0001, 0.05), 2)
        best_oracle = {"f1": -1.0, "tau0": np.nan, "tau2": np.nan, "thr": np.nan, "n_used": 0}
        for tau0 in tau_grid:
            for tau2 in tau_grid:
                if tau2 < tau0:
                    continue
                score_tau = oracle_rows["evidence_turnovers"].apply(
                    lambda turns: _row_score_with_taus(turns, tau0, tau2)
                ).astype(float).values
                valid = np.isfinite(score_tau)
                if valid.sum() == 0:
                    continue
                s = score_tau[valid]
                y = y_true_oracle[valid]
                uniq = np.unique(s)
                if len(uniq) <= 1:
                    continue
                cand_thr = ((uniq[:-1] + uniq[1:]) / 2.0).astype(float).tolist()
                for thr in cand_thr:
                    pred = np.where(s >= float(thr), 1, -1)
                    if np.unique(pred).size < 2:
                        continue
                    tp = int(((pred == 1) & (y == 1)).sum())
                    fp = int(((pred == 1) & (y == -1)).sum())
                    fn = int(((pred == -1) & (y == 1)).sum())
                    denom = (2 * tp) + fp + fn
                    f1 = float((2 * tp) / denom) if denom else 0.0
                    n_used = int(len(y))
                    if (f1 > best_oracle["f1"] + 1e-12) or (
                        abs(f1 - best_oracle["f1"]) <= 1e-12 and n_used > best_oracle["n_used"]
                    ):
                        best_oracle = {"f1": f1, "tau0": float(tau0), "tau2": float(tau2), "thr": float(thr), "n_used": n_used}

    if best_oracle["f1"] >= 0:
        out["Oracle upper bound f1"] = best_oracle["f1"]

    return out


def _two_group_local_density_max(pos_scores, neg_scores):
    from scipy.stats import gaussian_kde
    groups = [pos_scores, neg_scores]
    if len(pos_scores) == 0 or len(neg_scores) == 0:
        return np.nan

    y_lo = min(float(np.min(v)) for v in groups if len(v) > 0)
    y_hi = max(float(np.max(v)) for v in groups if len(v) > 0)
    if not np.isfinite(y_lo) or not np.isfinite(y_hi) or y_lo == y_hi:
        y_lo, y_hi = y_lo - 0.5, y_hi + 0.5
    y_grid = np.linspace(y_lo, y_hi, 256)

    dens_list = []
    for vals in groups:
        vals = np.asarray(vals, dtype=float)
        vals = vals[np.isfinite(vals)]
        if len(vals) >= 2 and np.std(vals) > 0 and np.unique(vals).size > 1:
            try:
                d = gaussian_kde(vals)(y_grid)
            except np.linalg.LinAlgError:
                bw = max((y_hi - y_lo) / 40.0, 1e-6)
                d = np.exp(-0.5 * ((y_grid - float(np.mean(vals))) / bw) ** 2)
        elif len(vals) == 1:
            bw = max((y_hi - y_lo) / 40.0, 1e-6)
            d = np.exp(-0.5 * ((y_grid - vals[0]) / bw) ** 2)
        else:
            d = np.zeros_like(y_grid)
        if len(vals) > 0:
            vmin, vmax = float(np.min(vals)), float(np.max(vals))
            d[(y_grid < vmin) | (y_grid > vmax)] = 0.0
        dens_list.append(d)

    local_max = max(float(np.max(d)) for d in dens_list) if dens_list else np.nan
    if not np.isfinite(local_max) or local_max <= 0:
        return np.nan
    return float(local_max)


def _draw_two_group_violin(ax, pos_scores, neg_scores, title, ylabel, y_refs, cats_local, density_max_override=None):
    if len(pos_scores) == 0 or len(neg_scores) == 0:
        ax.set_title(f"{title}\n[WARN] empty POS/NEG")
        ax.axis("off")
        return

    from scipy.stats import gaussian_kde
    groups = [pos_scores, neg_scores]
    y_lo = min(float(np.min(v)) for v in groups if len(v) > 0)
    y_hi = max(float(np.max(v)) for v in groups if len(v) > 0)
    if not np.isfinite(y_lo) or not np.isfinite(y_hi) or y_lo == y_hi:
        y_lo, y_hi = y_lo - 0.5, y_hi + 0.5
    y_grid = np.linspace(y_lo, y_hi, 256)

    dens_list = []
    for vals in groups:
        vals = np.asarray(vals, dtype=float)
        vals = vals[np.isfinite(vals)]
        if len(vals) >= 2 and np.std(vals) > 0 and np.unique(vals).size > 1:
            try:
                d = gaussian_kde(vals)(y_grid)
            except np.linalg.LinAlgError:
                bw = max((y_hi - y_lo) / 40.0, 1e-6)
                d = np.exp(-0.5 * ((y_grid - float(np.mean(vals))) / bw) ** 2)
        elif len(vals) == 1:
            bw = max((y_hi - y_lo) / 40.0, 1e-6)
            d = np.exp(-0.5 * ((y_grid - vals[0]) / bw) ** 2)
        else:
            d = np.zeros_like(y_grid)
        if len(vals) > 0:
            vmin, vmax = float(np.min(vals)), float(np.max(vals))
            d[(y_grid < vmin) | (y_grid > vmax)] = 0.0
        dens_list.append(d)

    local_max = max(float(np.max(d)) for d in dens_list) if dens_list else 0.0
    global_max = float(density_max_override) if (density_max_override is not None and np.isfinite(density_max_override) and density_max_override > 0) else local_max
    if not np.isfinite(global_max) or global_max <= 0:
        global_max = 1.0

    max_half_width = 0.35
    for i, (vals, dens) in enumerate(zip(groups, dens_list), start=1):
        w = (dens / global_max) * max_half_width
        ax.fill_betweenx(y_grid, i - w, i + w, color="C0", alpha=0.35, linewidth=0)
        vals = np.asarray(vals, dtype=float)
        vals = vals[np.isfinite(vals)]
        if len(vals) > 0:
            vmin, vmax = float(np.min(vals)), float(np.max(vals))
            med = float(np.median(vals))
            minmax_half = 0.08
            median_half = 0.14
            ax.plot([i, i], [vmin, vmax], color="grey", linewidth=1)
            ax.plot([i - minmax_half, i + minmax_half], [vmin, vmin], color="grey", linewidth=1.2)
            ax.plot([i - minmax_half, i + minmax_half], [vmax, vmax], color="grey", linewidth=1.2)
            ax.plot([i - median_half, i + median_half], [med, med], color="orange", linewidth=1.5)

    rng = np.random.default_rng(42)
    jitter = 0.04
    for i, vals in enumerate([pos_scores, neg_scores], start=1):
        x = i + rng.uniform(-jitter, jitter, size=len(vals))
        ax.scatter(x, vals, s=12, alpha=0.35, color="black", linewidths=0, zorder=1)

    ax.set_xlim(0.5, 2.5)
    ax.set_xticks([1, 2], labels=[f"POS ({cats_local[0]})", f"NEG ({cats_local[1]})"])
    for y_ref in y_refs:
        ax.axhline(y_ref, linestyle="--", linewidth=1)
    ax.set_title(title)
    ax.set_ylabel(ylabel)


def _draw_role_violin(ax, eval_df, pos_roles, neg_roles, title, xlabel, x_refs):
    from scipy.stats import gaussian_kde

    pos_roles_order = [r for r in pos_roles if r in set(eval_df["role"].astype(str))]
    neg_roles_order = [r for r in neg_roles if r in set(eval_df["role"].astype(str))]
    roles_plot_order = pos_roles_order + neg_roles_order
    if len(roles_plot_order) == 0:
        ax.set_title(f"{title}\n[WARN] no roles")
        ax.axis("off")
        return

    plot_df = eval_df[eval_df["role"].isin(roles_plot_order)].copy()
    data = [
        plot_df.loc[plot_df["role"] == r, "score_plot"].astype(float).dropna().values
        for r in roles_plot_order
    ]

    flat = np.concatenate([np.asarray(v, dtype=float) for v in data if len(v) > 0]) if any(len(v) > 0 for v in data) else np.array([0.0, 1.0])
    flat = flat[np.isfinite(flat)]
    if flat.size == 0:
        flat = np.array([0.0, 1.0])
    x_lo, x_hi = float(np.min(flat)), float(np.max(flat))
    if not np.isfinite(x_lo) or not np.isfinite(x_hi) or x_lo == x_hi:
        x_lo, x_hi = x_lo - 0.5, x_hi + 0.5
    x_grid = np.linspace(x_lo, x_hi, 256)

    dens_list = []
    for vals in data:
        vals = np.asarray(vals, dtype=float)
        vals = vals[np.isfinite(vals)]
        if len(vals) >= 2 and np.std(vals) > 0 and np.unique(vals).size > 1:
            try:
                d = gaussian_kde(vals)(x_grid)
            except np.linalg.LinAlgError:
                bw = max((x_hi - x_lo) / 40.0, 1e-6)
                d = np.exp(-0.5 * ((x_grid - float(np.mean(vals))) / bw) ** 2)
        elif len(vals) == 1:
            bw = max((x_hi - x_lo) / 40.0, 1e-6)
            d = np.exp(-0.5 * ((x_grid - vals[0]) / bw) ** 2)
        else:
            d = np.zeros_like(x_grid)
        if len(vals) > 0:
            vmin, vmax = float(np.min(vals)), float(np.max(vals))
            d[(x_grid < vmin) | (x_grid > vmax)] = 0.0
        dens_list.append(d)

    global_max = max(float(np.max(d)) for d in dens_list) if dens_list else 0.0
    if not np.isfinite(global_max) or global_max <= 0:
        global_max = 1.0

    group_maxes = np.array([float(np.max(d)) for d in dens_list]) if dens_list else np.array([global_max])
    nz = group_maxes[group_maxes > 0]
    density_ref = float(np.nanpercentile(nz, 85)) if nz.size > 0 else global_max
    if not np.isfinite(density_ref) or density_ref <= 0:
        density_ref = global_max

    width_scale = 0.35
    min_visible_half_width = 0.015
    gamma = 0.6
    width_list = []
    for dens in dens_list:
        dens_norm = dens / density_ref
        w = (dens_norm ** gamma) * width_scale
        w = np.where(dens > 0, np.maximum(w, min_visible_half_width), 0.0)
        width_list.append(w)

    half_max = np.array([float(np.max(w)) if w.size else 0.0 for w in width_list]) if width_list else np.array([])
    centers = np.zeros(len(data), dtype=float)
    if len(data) > 0:
        centers[0] = 1.0
        gap = 0.25
        min_step = 0.9
        for j in range(1, len(data)):
            step = float(half_max[j - 1] + half_max[j] + gap)
            centers[j] = centers[j - 1] + max(step, min_step)

    for c, vals, w in zip(centers, data, width_list):
        ax.fill_between(x_grid, c - w, c + w, color="C0", alpha=0.35, linewidth=0)
        vals = np.asarray(vals, dtype=float)
        vals = vals[np.isfinite(vals)]
        if len(vals) > 0:
            vmin, vmax = float(np.min(vals)), float(np.max(vals))
            med = float(np.median(vals))
            minmax_half = 0.16
            median_half = 0.28
            ax.plot([vmin, vmax], [c, c], color="grey", linewidth=1)
            ax.plot([vmin, vmin], [c - minmax_half, c + minmax_half], color="grey", linewidth=1)
            ax.plot([vmax, vmax], [c - minmax_half, c + minmax_half], color="grey", linewidth=1)
            ax.plot([med, med], [c - median_half, c + median_half], color="orange", linewidth=1.5)

    rng = np.random.default_rng(42)
    jitter = 0.08
    for c, vals in zip(centers, data):
        y = c + rng.uniform(-jitter, jitter, size=len(vals))
        ax.scatter(vals, y, s=10, alpha=0.3, color="black", linewidths=0, zorder=1)

    ax.set_yticks(centers, labels=roles_plot_order)
    if centers.size > 0:
        pad = float(np.max(half_max) + 0.25) if half_max.size > 0 else 0.5
        ax.set_ylim(float(centers[0] - pad), float(centers[-1] + pad))

    for x in x_refs:
        ax.axvline(x, linestyle="--", linewidth=1, color='grey')
    if len(pos_roles_order) > 0 and len(neg_roles_order) > 0:
        i_pos_last = len(pos_roles_order) - 1
        i_neg_first = len(pos_roles_order)
        sep_y = (float(centers[i_pos_last]) + float(centers[i_neg_first])) / 2.0
        ax.axhline(sep_y, linestyle=":", linewidth=1, color='black')

    ax.set_title(title)
    ax.set_xlabel(xlabel)


for IDX_RUN in IDX_LIST:
    pos_roles, neg_roles = _roles_for_idx(IDX_RUN)
    roles_of_interest = set(pos_roles + neg_roles)

    project_root = infer_project_root(RUN_ROOT, SUBFOLDERS)
    run_root_name = RUN_ROOT.name
    q = _resolve_question(project_root, run_root_name, IDX_RUN, SUBFOLDERS)
    cats_local = q.get("related_categories", [])
    if int(IDX_RUN) == 9 and len(cats_local) >= 3:
        pos_cat = [cats_local[0], cats_local[1]]
        neg_cat = cats_local[2]
        pos_cat_tick = f"{cats_local[0]} + {cats_local[1]}"
        neg_cat_tick = str(cats_local[2])
    else:
        pos_cat = cats_local[0] if len(cats_local) >= 1 else None
        neg_cat = cats_local[1] if len(cats_local) >= 2 else None
        pos_cat_tick = str(cats_local[0]) if len(cats_local) >= 1 else "POS"
        neg_cat_tick = str(cats_local[1]) if len(cats_local) >= 2 else "NEG"

    print("\n" + "#" * 120)
    print(f"IDX {IDX_RUN} QUESTION:")
    print(q.get("question", ""))
    print("CATEGORIES:", cats_local)

    model_specs = list(SUBFOLDERS.items())
    model_order_local = []
    wide_by_model = {}
    pre_raw_by_model = {}
    pre_by_model = {}
    post_by_model = {}
    score_label_by_model = {}
    plot_label_by_model = {}
    plot_refs_by_model = {}

    for subfolder, model_name in model_specs:
        csv_path = resolve_csv_path(project_root, run_root_name, subfolder, model_name, IDX_RUN)
        if not csv_path.exists():
            continue
        df_wide = pd.read_csv(csv_path, dtype=str, low_memory=False)
        eval_df = to_long_eval_df(df_wide, roles_of_interest=roles_of_interest)
        eval_df_raw = eval_df.copy()

        if use_prob:
            eval_df["prob_used"] = eval_df["evidence_turnovers"].apply(_row_signed_prob_direct_summary).astype(float)
            score_col = "prob_used"
            score_label = f"signed_probability (thr={prob_decision_threshold_summary})"
        else:
            score_col = "avg_score"
            score_label = "avg_score"
            if score_col not in eval_df.columns:
                eval_df[score_col] = np.nan

        eval_df["score_used"] = pd.to_numeric(eval_df[score_col], errors="coerce")
        if use_prob:
            score_arr = pd.to_numeric(eval_df["score_used"], errors="coerce").values.astype(float)
            pred_arr = np.where(score_arr >= float(prob_decision_threshold_summary), 1, -1)
            eval_df["pred_pole"] = np.where(np.isfinite(score_arr), pred_arr, 0).astype(int)
        else:
            eval_df["pred_pole"] = eval_df["score_used"].apply(sign3)

        if use_prob:
            eval_df["score_plot"] = pd.to_numeric(eval_df["score_used"], errors="coerce")
            plot_label = f"{score_label}"
            plot_refs = [-1, -0.5, 0, 0.5, 1]
        else:
            eval_df["score_plot"] = pd.to_numeric(eval_df["score_used"], errors="coerce")
            plot_label = score_label
            plot_refs = [-2, -1, 0, 1, 2]

        wide_by_model[model_name] = df_wide
        pre_raw_by_model[model_name] = eval_df_raw
        pre_by_model[model_name] = eval_df
        score_label_by_model[model_name] = score_label
        plot_label_by_model[model_name] = plot_label
        plot_refs_by_model[model_name] = plot_refs
        model_order_local.append(model_name)

    if filter_by_lowest and len(model_order_local) > 0:
        key_sets = []
        for model_name in model_order_local:
            dd = pre_by_model[model_name]
            if len(dd) == 0:
                key_sets.append(set())
                continue
            if {"a_id", "b_id", "speaker"}.issubset(dd.columns):
                rk = set(zip(dd["a_id"].astype(str), dd["b_id"].astype(str), dd["speaker"].astype(str)))
            elif {"prompt_id_unique", "speaker"}.issubset(dd.columns):
                rk = set(zip(dd["prompt_id_unique"].astype(str), dd["speaker"].astype(str)))
            else:
                rk = set()
            key_sets.append(rk)
        common_keys = set.intersection(*key_sets) if key_sets else set()

        for model_name in model_order_local:
            dd = pre_by_model[model_name]
            if len(dd) == 0:
                pre_by_model[model_name] = dd.copy()
                continue
            if {"a_id", "b_id", "speaker"}.issubset(dd.columns):
                rk = pd.Series(list(zip(dd["a_id"].astype(str), dd["b_id"].astype(str), dd["speaker"].astype(str))), index=dd.index)
            elif {"prompt_id_unique", "speaker"}.issubset(dd.columns):
                rk = pd.Series(list(zip(dd["prompt_id_unique"].astype(str), dd["speaker"].astype(str))), index=dd.index)
            else:
                pre_by_model[model_name] = dd.iloc[0:0].copy()
                continue
            pre_by_model[model_name] = dd.loc[rk.isin(common_keys)].copy()

    metadata_rows = []
    result_rows = []

    for model_name in model_order_local:
        df_wide = wide_by_model[model_name]
        eval_pre = pre_by_model[model_name].copy()
        eval_raw = pre_raw_by_model[model_name].copy()

        # failure_rate: always from raw CSV attempted slots (before any notebook-side filtering)
        # attempted slot = any non-empty avg_score OR fail_note OR evidence for that speaker side.
        attempted_slots = 0
        success_slots = 0
        for _, r in df_wide.iterrows():
            for spk in ["a", "b"]:
                avg_s = str(r.get(f"avg_score_{spk}", "")).strip()
                fail_s = str(r.get(f"fail_note_{spk}", "")).strip()
                evid_s = str(r.get(f"evidence_{spk}", "")).strip()

                avg_ok = avg_s not in ("", "nan", "NaN")
                attempted = avg_ok or (fail_s not in ("", "nan", "NaN")) or (evid_s not in ("", "nan", "NaN"))
                if attempted:
                    attempted_slots += 1
                if avg_ok:
                    success_slots += 1

        failure_slots = attempted_slots - success_slots
        failure_rate = (failure_slots / attempted_slots) if attempted_slots > 0 else np.nan

        mean_item_dis = float(pd.to_numeric(eval_pre["disagreement_item_rate"], errors="coerce").dropna().mean()) if len(eval_pre) else np.nan

        eval_post = eval_pre.copy()
        if exclude_variant_disagree and len(eval_post):
            rate = pd.to_numeric(eval_post.get("disagreement_item_rate", pd.Series(np.nan, index=eval_post.index)), errors="coerce")
            exclude_by_rate = rate >= disagree_rate_exclude_thresh
            exclude_by_row_level = eval_post.get("row_level_variant_disagreement", pd.Series(False, index=eval_post.index)).fillna(False).astype(bool)
            exclude_mask = exclude_by_rate | exclude_by_row_level
            eval_post = eval_post[~exclude_mask].copy()

        post_by_model[model_name] = eval_post

        metadata_rows.append({
            "Model": model_name,
            "Conversations (rows in CSV)": int(len(df_wide)),
            "Evaluated speakers (pre filter)": int(len(eval_pre)),
            "failure_rate(regardless of filter_by_lowest)": float(failure_rate),
            "Mean item disagreement rate": float(mean_item_dis) if np.isfinite(mean_item_dis) else np.nan,
            "Evaluated speakers (post filter)": int(len(eval_post)),
        })

        perf = _compute_perf_metrics(eval_post, pos_cat, neg_cat, use_prob)
        result_rows.append({"Model": model_name, **perf})

    md_df = pd.DataFrame(metadata_rows)
    rs_df = pd.DataFrame(result_rows)

    print("\nMetadata table")
    display(md_df)
    print("\nResults table")
    display(rs_df)

    # 1 x n merged two-group violin plots
    n_models = len(model_order_local)
    if n_models > 0:
        category_density_max_global = None
        if absolute_category_violin:
            density_max_candidates = []
            for model_name in model_order_local:
                dd = post_by_model[model_name]
                pos_scores = dd.loc[dd["role"].isin(pos_roles), "score_plot"].astype(float).dropna().values
                neg_scores = dd.loc[dd["role"].isin(neg_roles), "score_plot"].astype(float).dropna().values
                local_density_max = _two_group_local_density_max(pos_scores, neg_scores)
                if np.isfinite(local_density_max) and local_density_max > 0:
                    density_max_candidates.append(float(local_density_max))
            if len(density_max_candidates) > 0:
                category_density_max_global = float(np.max(density_max_candidates))

        fig, axes = plt.subplots(1, n_models, figsize=(5 * n_models, 4))
        if n_models == 1:
            axes = [axes]
        for ax, model_name in zip(axes, model_order_local):
            dd = post_by_model[model_name]
            pos_scores = dd.loc[dd["role"].isin(pos_roles), "score_plot"].astype(float).dropna().values
            neg_scores = dd.loc[dd["role"].isin(neg_roles), "score_plot"].astype(float).dropna().values
            _draw_two_group_violin(
                ax,
                pos_scores,
                neg_scores,
                f"{model_name}: POS vs NEG",
                plot_label_by_model[model_name],
                plot_refs_by_model[model_name],
                [pos_cat_tick, neg_cat_tick],
                density_max_override=category_density_max_global if absolute_category_violin else None,
            )
        plt.tight_layout()
        plt.show()

    # ROC: all models in one plot
    if n_models > 0:
        fig, ax = plt.subplots(figsize=(6, 5))
        any_curve = False
        for model_name in model_order_local:
            dd = post_by_model[model_name]
            merged = dd[dd["role"].isin(set(pos_roles) | set(neg_roles))].copy()
            merged = merged.dropna(subset=["score_used"])
            merged["y_true"] = merged["role"].isin(pos_roles).astype(int)
            y_true = merged["y_true"].values
            y_score = merged["score_used"].astype(float).values
            if len(np.unique(y_true)) < 2:
                continue
            try:
                from sklearn.metrics import roc_curve, roc_auc_score
                fpr, tpr, _ = roc_curve(y_true, y_score)
                auc_val = roc_auc_score(y_true, y_score)
            except Exception:
                pos = y_score[y_true == 1]
                neg = y_score[y_true == 0]
                if len(pos) == 0 or len(neg) == 0:
                    continue
                u = mannwhitneyu(pos, neg, alternative="two-sided")
                auc_val = float(u.statistic / (len(pos) * len(neg)))
                thr = np.sort(np.unique(y_score))
                fpr, tpr = [], []
                for t in thr:
                    pred = (y_score >= t).astype(int)
                    tp = ((pred == 1) & (y_true == 1)).sum()
                    fp = ((pred == 1) & (y_true == 0)).sum()
                    fn = ((pred == 0) & (y_true == 1)).sum()
                    tn = ((pred == 0) & (y_true == 0)).sum()
                    tpr.append(tp / (tp + fn) if (tp + fn) else 0.0)
                    fpr.append(fp / (fp + tn) if (fp + tn) else 0.0)
                fpr = np.array(fpr)
                tpr = np.array(tpr)
            any_curve = True
            (line,) = ax.plot(fpr, tpr, linewidth=2, label=f"{model_name} (AUC={auc_val:.3f})")
            ax.fill_between(fpr, tpr, 0, alpha=0.15, color=line.get_color())

        ax.plot([0, 1], [0, 1], linestyle="--", linewidth=1, color="gray")
        ax.set_title(f"IDX {IDX_RUN}: ROC across models")
        ax.set_xlabel("False Positive Rate")
        ax.set_ylabel("True Positive Rate")
        if any_curve:
            ax.legend(loc="lower right")
        plt.tight_layout()
        plt.show()

    # n x 1 role-specific violin subplot
    if n_models > 0:
        max_roles = max(1, len(pos_roles) + len(neg_roles))
        fig_w = max(10, 0.8 * max_roles)
        fig_h = 4 * n_models
        fig, axes = plt.subplots(n_models, 1, figsize=(fig_w, fig_h))
        if n_models == 1:
            axes = [axes]
        for ax, model_name in zip(axes, model_order_local):
            _draw_role_violin(
                ax,
                post_by_model[model_name],
                pos_roles,
                neg_roles,
                f"{model_name}: role-specific score distribution",
                plot_label_by_model[model_name],
                plot_refs_by_model[model_name],
            )
        plt.tight_layout()
        plt.show()

In [ ]:
# Aggregated multi-question summary (single metadata table + single results table)
# Parameters
idx_list = [0, 1, 2, 3, 4, 5, 6, 7, 8]
display_idx_shift_map = {}

def _display_idx_shift(v):
    try:
        i = int(v)
    except Exception:
        return v
    return display_idx_shift_map.get(i, i)

model_selection = {
    "models/qwen_omni": "Qwen",
    "models/kimi_audio": "Kimi",
    "models/granite_speech": "Granite",
    "models/gpt_audio": "GPT",
    "models/gemini_audio": "Gemini",
}  # subfolder -> model name (for plots/tables); set to None to use SUBFOLDERS

# model_selection = {
#     "models/qwen_omni": "Qwen",
#     "models/qwen_transcript_ablation": "Qwen transcript ablation",
# }  # subfolder -> model name (for plots/tables); set to None to use SUBFOLDERS

meta_table_col = [
    "IDX",
    # "Question",
    "Model",
    "Conversations (rows in CSV)",
    "Evaluated speakers (pre filter)",
    "failure_rate(regardless of filter_by_lowest)",
    "Mean item disagreement rate",
    # "Evaluated speakers (post filter)",
]
result_table_col = [
    "IDX",
    # "Question",
    "Model",    
    "failure_rate(regardless of filter_by_lowest)",
    "Mean item disagreement rate",
    "Categorical pole-consistency",
    "Oracle upper bound f1",
    "Equal Error Rate",
    "AUROC",
]

merge_table = True

merged_col = [
    "IDX",
    "Model",
    # "Conversations (rows in CSV)",
    "Evaluated speakers (pre filter)",
    "failure_rate(regardless of filter_by_lowest)",
    "Mean item disagreement rate",
    # "Evaluated speakers (post filter)",
    "Categorical pole-consistency",
    "Oracle upper bound f1",
    "Equal Error Rate",
    "AUROC",
]


_use_prob = bool(globals().get("use_prob", False))
_filter_by_lowest = bool(globals().get("filter_by_lowest", False))
_exclude_variant_disagree = bool(globals().get("exclude_variant_disagree", False))
_disagree_rate_exclude_thresh = float(globals().get("disagree_rate_exclude_thresh", 0.2))
_prob_thr = float(globals().get("prob_decision_threshold_summary", globals().get("prob_decision_threshold", 0.0)))

# Keep this cell runnable in a fresh kernel: define missing helpers locally.

# Fallback helper defs (if the previous summary cell helper defs were not executed)
if "_coerce_prob_item_summary" not in globals():
    def _coerce_prob_item_summary(item, vidx):
        p = safe_float(item.get(f"prob_v{vidx}", np.nan))
        if not np.isfinite(p):
            p = safe_float(item.get(f"probablity_v{vidx}", np.nan))
        if np.isfinite(p) and p > 1.0:
            p = p / 100.0
        if np.isfinite(p):
            p = float(np.clip(p, 0.0, 1.0))
        return p

if "_item_signed_prob_summary" not in globals():
    def _item_signed_prob_summary(choice, prob):
        c = str(choice).strip().upper()
        if c not in {"P", "N"} or not np.isfinite(prob):
            return np.nan
        return float(prob) if c == "P" else float(-prob)

if "_row_signed_prob_direct_summary" not in globals():
    def _row_signed_prob_direct_summary(turns):
        vals = []
        if not isinstance(turns, list):
            return np.nan
        for t in turns:
            p0 = _coerce_prob_item_summary(t, 0)
            p1 = _coerce_prob_item_summary(t, 1)
            q0 = _item_signed_prob_summary(t.get("choice_v0", None), p0)
            q1 = _item_signed_prob_summary(t.get("choice_v1", None), p1)
            if not np.isfinite(q0) or not np.isfinite(q1):
                continue
            vals.append((float(q0) + float(q1)) / 2.0)
        return float(np.mean(vals)) if vals else np.nan

if "_compute_perf_metrics" not in globals():
    def _compute_perf_metrics(eval_df, pos_category, neg_category, use_prob_flag):
        out = {
            "Categorical pole-consistency": np.nan,
            "Equal Error Rate": np.nan,
            "AUROC": np.nan,
            "Oracle upper bound f1": np.nan,
        }
        if len(eval_df) == 0 or ("category" not in eval_df.columns):
            return out

        pos_set = set(pos_category) if isinstance(pos_category, (list, tuple, set)) else ({pos_category} if pos_category is not None else set())
        neg_set = set(neg_category) if isinstance(neg_category, (list, tuple, set)) else ({neg_category} if neg_category is not None else set())

        def expected_cat_pole(cat):
            if cat in pos_set:
                return 1
            if cat in neg_set:
                return -1
            return 0

        dd = eval_df.copy()
        dd["exp_cat_pole"] = dd["category"].apply(expected_cat_pole)
        mask = (dd["exp_cat_pole"] != 0) & (dd["pred_pole"] != 0)
        out["Categorical pole-consistency"] = (dd.loc[mask, "pred_pole"] == dd.loc[mask, "exp_cat_pole"]).mean()

        y_true_cat = (dd.loc[mask, "exp_cat_pole"] == 1).astype(int).values
        y_score_cat = dd.loc[mask, "score_used"].astype(float).values
        if len(y_true_cat) > 0 and len(np.unique(y_true_cat)) == 2:
            try:
                from sklearn.metrics import roc_curve, roc_auc_score
                fpr_cat, tpr_cat, _ = roc_curve(y_true_cat, y_score_cat)
                fnr_cat = 1.0 - tpr_cat
                idx_cat = int(np.nanargmin(np.abs(fnr_cat - fpr_cat)))
                out["Equal Error Rate"] = float((fpr_cat[idx_cat] + fnr_cat[idx_cat]) / 2.0)
                out["AUROC"] = float(roc_auc_score(y_true_cat, y_score_cat))
            except Exception:
                thr_cat = np.sort(np.unique(y_score_cat))
                fpr_cat, fnr_cat, tpr_cat = [], [], []
                for t in thr_cat:
                    pred = (y_score_cat >= t).astype(int)
                    tp = ((pred == 1) & (y_true_cat == 1)).sum()
                    fp = ((pred == 1) & (y_true_cat == 0)).sum()
                    fn = ((pred == 0) & (y_true_cat == 1)).sum()
                    tn = ((pred == 0) & (y_true_cat == 0)).sum()
                    fpr_cat.append(fp / (fp + tn) if (fp + tn) else 0.0)
                    fnr_cat.append(fn / (fn + tp) if (fn + tp) else 0.0)
                    tpr_cat.append(tp / (tp + fn) if (tp + fn) else 0.0)
                fpr_cat = np.array(fpr_cat)
                fnr_cat = np.array(fnr_cat)
                tpr_cat = np.array(tpr_cat)
                idx_cat = int(np.nanargmin(np.abs(fnr_cat - fpr_cat)))
                out["Equal Error Rate"] = float((fpr_cat[idx_cat] + fnr_cat[idx_cat]) / 2.0)
                if len(fpr_cat) > 1:
                    order = np.argsort(fpr_cat)
                    out["AUROC"] = float(np.trapz(tpr_cat[order], fpr_cat[order]))

        eligible_cat = (dd["exp_cat_pole"] != 0)

        def _coerce_prob_item(item, vidx):
            return _coerce_prob_item_summary(item, vidx)

        def _item_signed_prob(choice, prob):
            return _item_signed_prob_summary(choice, prob)

        def _row_prob_direct(turns):
            vals = []
            if not isinstance(turns, list):
                return np.nan
            for t in turns:
                c0 = t.get("choice_v0", None)
                c1 = t.get("choice_v1", None)
                p0 = _coerce_prob_item(t, 0)
                p1 = _coerce_prob_item(t, 1)
                q0 = _item_signed_prob(c0, p0)
                q1 = _item_signed_prob(c1, p1)
                if not np.isfinite(q0) or not np.isfinite(q1):
                    continue
                vals.append((float(q0) + float(q1)) / 2.0)
            return float(np.mean(vals)) if vals else np.nan

        def _map_choice_prob(choice, prob, tau0, tau2):
            c = str(choice).strip().upper()
            if c not in {"P", "N"} or not np.isfinite(prob):
                return None
            s = 1 if c == "P" else -1
            if prob < float(tau0):
                return 0
            if prob >= float(tau2):
                return 2 * s
            return 1 * s

        def _row_score_with_taus(turns, tau0, tau2):
            vals = []
            if not isinstance(turns, list):
                return np.nan
            for t in turns:
                c0 = t.get("choice_v0", None)
                c1 = t.get("choice_v1", None)
                p0 = _coerce_prob_item(t, 0)
                p1 = _coerce_prob_item(t, 1)
                s0 = _map_choice_prob(c0, p0, tau0, tau2)
                s1 = _map_choice_prob(c1, p1, tau0, tau2)
                if s0 is None or s1 is None:
                    continue
                vals.append((float(s0) + float(s1)) / 2.0)
            return float(np.mean(vals)) if vals else np.nan

        oracle_rows = dd.loc[eligible_cat, ["exp_cat_pole", "evidence_turnovers"]].copy()
        y_true_oracle = oracle_rows["exp_cat_pole"].astype(int).values

        if use_prob_flag:
            best_oracle = {"f1": -1.0, "thr": np.nan, "n_used": 0}
            prob_signal = oracle_rows["evidence_turnovers"].apply(_row_prob_direct).astype(float).values
            valid = np.isfinite(prob_signal)
            if valid.sum() > 0:
                s = prob_signal[valid]
                y = y_true_oracle[valid]
                uniq = np.unique(s)
                if len(uniq) > 1:
                    cand_thr = ((uniq[:-1] + uniq[1:]) / 2.0).astype(float).tolist()
                    for thr in cand_thr:
                        pred = np.where(s >= float(thr), 1, -1)
                        if np.unique(pred).size < 2:
                            continue
                        tp = int(((pred == 1) & (y == 1)).sum())
                        fp = int(((pred == 1) & (y == -1)).sum())
                        fn = int(((pred == -1) & (y == 1)).sum())
                        denom = (2 * tp) + fp + fn
                        f1 = float((2 * tp) / denom) if denom else 0.0
                        n_used = int(len(y))
                        if (f1 > best_oracle["f1"] + 1e-12) or (
                            abs(f1 - best_oracle["f1"]) <= 1e-12 and n_used > best_oracle["n_used"]
                        ):
                            best_oracle = {"f1": f1, "thr": float(thr), "n_used": n_used}
        else:
            tau_grid = np.round(np.arange(0.0, 1.0001, 0.05), 2)
            best_oracle = {"f1": -1.0, "tau0": np.nan, "tau2": np.nan, "thr": np.nan, "n_used": 0}
            for tau0 in tau_grid:
                for tau2 in tau_grid:
                    if tau2 < tau0:
                        continue
                    score_tau = oracle_rows["evidence_turnovers"].apply(
                        lambda turns: _row_score_with_taus(turns, tau0, tau2)
                    ).astype(float).values
                    valid = np.isfinite(score_tau)
                    if valid.sum() == 0:
                        continue
                    s = score_tau[valid]
                    y = y_true_oracle[valid]
                    uniq = np.unique(s)
                    if len(uniq) <= 1:
                        continue
                    cand_thr = ((uniq[:-1] + uniq[1:]) / 2.0).astype(float).tolist()
                    for thr in cand_thr:
                        pred = np.where(s >= float(thr), 1, -1)
                        if np.unique(pred).size < 2:
                            continue
                        tp = int(((pred == 1) & (y == 1)).sum())
                        fp = int(((pred == 1) & (y == -1)).sum())
                        fn = int(((pred == -1) & (y == 1)).sum())
                        denom = (2 * tp) + fp + fn
                        f1 = float((2 * tp) / denom) if denom else 0.0
                        n_used = int(len(y))
                        if (f1 > best_oracle["f1"] + 1e-12) or (
                            abs(f1 - best_oracle["f1"]) <= 1e-12 and n_used > best_oracle["n_used"]
                        ):
                            best_oracle = {"f1": f1, "tau0": float(tau0), "tau2": float(tau2), "thr": float(thr), "n_used": n_used}

        if best_oracle["f1"] >= 0:
            out["Oracle upper bound f1"] = best_oracle["f1"]

        return out

if "_draw_two_group_violin" not in globals():
    def _draw_two_group_violin(ax, pos_scores, neg_scores, title, ylabel, y_refs, cats_local, density_max_override=None):
        if len(pos_scores) == 0 or len(neg_scores) == 0:
            ax.set_title(f"{title}\n[WARN] empty POS/NEG")
            ax.axis("off")
            return

        from scipy.stats import gaussian_kde
        groups = [pos_scores, neg_scores]
        y_lo = min(float(np.min(v)) for v in groups if len(v) > 0)
        y_hi = max(float(np.max(v)) for v in groups if len(v) > 0)
        if not np.isfinite(y_lo) or not np.isfinite(y_hi) or y_lo == y_hi:
            y_lo, y_hi = y_lo - 0.5, y_hi + 0.5
        y_grid = np.linspace(y_lo, y_hi, 256)

        dens_list = []
        for vals in groups:
            vals = np.asarray(vals, dtype=float)
            vals = vals[np.isfinite(vals)]
            if len(vals) >= 2 and np.std(vals) > 0 and np.unique(vals).size > 1:
                try:
                    d = gaussian_kde(vals)(y_grid)
                except np.linalg.LinAlgError:
                    bw = max((y_hi - y_lo) / 40.0, 1e-6)
                    d = np.exp(-0.5 * ((y_grid - float(np.mean(vals))) / bw) ** 2)
            elif len(vals) == 1:
                bw = max((y_hi - y_lo) / 40.0, 1e-6)
                d = np.exp(-0.5 * ((y_grid - vals[0]) / bw) ** 2)
            else:
                d = np.zeros_like(y_grid)
            if len(vals) > 0:
                vmin, vmax = float(np.min(vals)), float(np.max(vals))
                d[(y_grid < vmin) | (y_grid > vmax)] = 0.0
            dens_list.append(d)

        local_max = max(float(np.max(d)) for d in dens_list) if dens_list else 0.0
        global_max = float(density_max_override) if (density_max_override is not None and np.isfinite(density_max_override) and density_max_override > 0) else local_max
        if not np.isfinite(global_max) or global_max <= 0:
            global_max = 1.0

        max_half_width = 0.35
        for i, (vals, dens) in enumerate(zip(groups, dens_list), start=1):
            w = (dens / global_max) * max_half_width
            ax.fill_betweenx(y_grid, i - w, i + w, color="C0", alpha=0.35, linewidth=0)
            vals = np.asarray(vals, dtype=float)
            vals = vals[np.isfinite(vals)]
            if len(vals) > 0:
                vmin, vmax = float(np.min(vals)), float(np.max(vals))
                med = float(np.median(vals))
                minmax_half = 0.08
                median_half = 0.14
                ax.plot([i, i], [vmin, vmax], color="grey", linewidth=1)
                ax.plot([i - minmax_half, i + minmax_half], [vmin, vmin], color="grey", linewidth=1.2)
                ax.plot([i - minmax_half, i + minmax_half], [vmax, vmax], color="grey", linewidth=1.2)
                ax.plot([i - median_half, i + median_half], [med, med], color="orange", linewidth=1.5)

        rng = np.random.default_rng(42)
        jitter = 0.04
        for i, vals in enumerate([pos_scores, neg_scores], start=1):
            x = i + rng.uniform(-jitter, jitter, size=len(vals))
            ax.scatter(x, vals, s=12, alpha=0.35, color="black", linewidths=0, zorder=1)

        ax.set_xlim(0.5, 2.5)
        ax.set_xticks([1, 2], labels=[f"POS ({cats_local[0]})", f"NEG ({cats_local[1]})"])
        for y_ref in y_refs:
            ax.axhline(y_ref, linestyle="--", linewidth=1)
        ax.set_title(title)
        ax.set_ylabel(ylabel)


def _select_cols(df, cols, table_name):
    if cols is None:
        return df
    keep = [c for c in cols if c in df.columns]
    missing = [c for c in cols if c not in df.columns]
    if missing:
        print(f"[WARN] {table_name}: missing columns skipped -> {missing}")
    return df.loc[:, keep]


BEST_METRIC_DIRECTION = {
    "failure_rate(regardless of filter_by_lowest)": "min",
    "Mean item disagreement rate": "min",
    "Equal Error Rate": "min",
    "Categorical pole-consistency": "max",
    "Oracle upper bound f1": "max",
    "AUROC": "max",
}


def _format_summary_cell(v):
    if pd.isna(v):
        return ""

    if isinstance(v, str):
        vv = v.strip()
        if vv == "-":
            return "-"
        try:
            fv = float(vv)
        except Exception:
            return v
    else:
        try:
            fv = float(v)
        except Exception:
            return str(v)

    if not np.isfinite(fv):
        return str(v)
    if float(fv).is_integer() and abs(fv) >= 1:
        return f"{int(fv)}"
    if 0 < abs(fv) < 0.01:
        return f"{fv:.3f}"
    return f"{fv:.2f}"


def _best_metric_styles(df_source, df_show):
    styles = pd.DataFrame("", index=df_show.index, columns=df_show.columns)
    if "IDX" not in df_source.columns:
        return styles

    idx_vals = pd.to_numeric(df_source["IDX"], errors="coerce")
    for metric, direction in BEST_METRIC_DIRECTION.items():
        if metric not in df_source.columns or metric not in df_show.columns:
            continue

        metric_vals = pd.to_numeric(df_source[metric], errors="coerce")
        finite_mask = pd.Series(np.isfinite(metric_vals.to_numpy(dtype=float)), index=metric_vals.index)

        for q in idx_vals.dropna().unique():
            mask_q = idx_vals == q
            vals_q = metric_vals[mask_q & finite_mask]
            if len(vals_q) == 0:
                continue

            best = float(vals_q.min()) if direction == "min" else float(vals_q.max())
            is_best = pd.Series(
                np.isclose(metric_vals.to_numpy(dtype=float), best, rtol=1e-9, atol=1e-12),
                index=metric_vals.index,
            )
            best_mask = mask_q & finite_mask & is_best
            styles.loc[best_mask, metric] = "background-color: #fff3a3; font-weight: 700;"

    return styles


def _display_styled_summary_table(title, df_source, df_show):
    print(f"\n{title}")
    style_df = _best_metric_styles(df_source, df_show)
    styler = df_show.style.format(_format_summary_cell).apply(lambda _df: style_df, axis=None)
    display(styler)


project_root = infer_project_root(RUN_ROOT, SUBFOLDERS)
run_root_name = RUN_ROOT.name

selected_subfolders = model_selection if model_selection is not None else dict(SUBFOLDERS)
if not isinstance(selected_subfolders, dict) or len(selected_subfolders) == 0:
    raise ValueError("model_selection must be a non-empty dict (or None to use SUBFOLDERS).")

model_specs = list(selected_subfolders.items())  # (subfolder, model_name)
model_order_master = [m for _, m in model_specs]

def _is_placeholder_subfolder_name(name):
    s = str(name).strip().lower().replace("_", " ")
    return s.startswith("placeholder") or s.startswith("place holder")

placeholder_models = {
    model_name
    for subfolder, model_name in model_specs
    if _is_placeholder_subfolder_name(subfolder)
}


metadata_rows = []
result_rows = []
post_by_q_model = {}
plot_label_by_q_model = {}
plot_refs_by_q_model = {}
roles_by_q = {}
cats_by_q = {}

for IDX_RUN in idx_list:
    pos_roles, neg_roles = _roles_for_idx(IDX_RUN)
    roles_of_interest = set(pos_roles + neg_roles)
    roles_by_q[IDX_RUN] = (pos_roles, neg_roles)

    qpath = _resolve_question_path(project_root, run_root_name, IDX_RUN, selected_subfolders)
    q = load_question(qpath)
    question_text = q.get("question", "")
    cats_local = q.get("related_categories", [])
    if int(IDX_RUN) == 9 and len(cats_local) >= 3:
        pos_cat = [cats_local[0], cats_local[1]]
        neg_cat = cats_local[2]
        pos_cat_tick = f"{cats_local[0]} + {cats_local[1]}"
        neg_cat_tick = str(cats_local[2])
    else:
        pos_cat = cats_local[0] if len(cats_local) >= 1 else None
        neg_cat = cats_local[1] if len(cats_local) >= 2 else None
        pos_cat_tick = str(cats_local[0]) if len(cats_local) >= 1 else "POS"
        neg_cat_tick = str(cats_local[1]) if len(cats_local) >= 2 else "NEG"
    cats_by_q[IDX_RUN] = cats_local

    pre_by_model = {}
    wide_by_model = {}

    for subfolder, model_name in model_specs:
        if model_name in placeholder_models:
            continue

        csv_path = resolve_csv_path(project_root, run_root_name, subfolder, model_name, IDX_RUN)
        if not csv_path.exists():
            continue

        df_wide = pd.read_csv(csv_path, dtype=str, low_memory=False)
        eval_df = to_long_eval_df(df_wide, roles_of_interest=roles_of_interest)

        # Keep preprocessing identical to the per-question summary cell
        if _use_prob:
            eval_df["prob_used"] = eval_df["evidence_turnovers"].apply(_row_signed_prob_direct_summary).astype(float)
            score_col = "prob_used"
            score_label = f"signed_probability (thr={_prob_thr})"
        else:
            score_col = "avg_score"
            score_label = "avg_score"
            if score_col not in eval_df.columns:
                eval_df[score_col] = np.nan

        eval_df["score_used"] = pd.to_numeric(eval_df[score_col], errors="coerce")
        if _use_prob:
            score_arr = pd.to_numeric(eval_df["score_used"], errors="coerce").values.astype(float)
            pred_arr = np.where(score_arr >= float(_prob_thr), 1, -1)
            eval_df["pred_pole"] = np.where(np.isfinite(score_arr), pred_arr, 0).astype(int)
            eval_df["score_plot"] = pd.to_numeric(eval_df["score_used"], errors="coerce")
            plot_refs = [-1, -0.5, 0, 0.5, 1]
        else:
            eval_df["pred_pole"] = eval_df["score_used"].apply(sign3)
            eval_df["score_plot"] = pd.to_numeric(eval_df["score_used"], errors="coerce")
            plot_refs = [-2, -1, 0, 1, 2]

        pre_by_model[model_name] = eval_df
        wide_by_model[model_name] = df_wide
        plot_label_by_q_model[(IDX_RUN, model_name)] = score_label
        plot_refs_by_q_model[(IDX_RUN, model_name)] = plot_refs

    # Optional fairness filter: intersection of available rows across models for this question
    if _filter_by_lowest and len(pre_by_model) > 0:
        model_names_local = [m for m in model_order_master if m in pre_by_model]
        key_sets = []
        for model_name in model_names_local:
            dd = pre_by_model[model_name]
            if len(dd) == 0:
                key_sets.append(set())
                continue
            if {"a_id", "b_id", "speaker"}.issubset(dd.columns):
                rk = set(zip(dd["a_id"].astype(str), dd["b_id"].astype(str), dd["speaker"].astype(str)))
            elif {"prompt_id_unique", "speaker"}.issubset(dd.columns):
                rk = set(zip(dd["prompt_id_unique"].astype(str), dd["speaker"].astype(str)))
            else:
                rk = set()
            key_sets.append(rk)
        common_keys = set.intersection(*key_sets) if key_sets else set()

        for model_name in model_names_local:
            dd = pre_by_model[model_name]
            if len(dd) == 0:
                pre_by_model[model_name] = dd.copy()
                continue
            if {"a_id", "b_id", "speaker"}.issubset(dd.columns):
                rk = pd.Series(list(zip(dd["a_id"].astype(str), dd["b_id"].astype(str), dd["speaker"].astype(str))), index=dd.index)
            elif {"prompt_id_unique", "speaker"}.issubset(dd.columns):
                rk = pd.Series(list(zip(dd["prompt_id_unique"].astype(str), dd["speaker"].astype(str))), index=dd.index)
            else:
                pre_by_model[model_name] = dd.iloc[0:0].copy()
                continue
            pre_by_model[model_name] = dd.loc[rk.isin(common_keys)].copy()

    # Per-model rows (include all selected models; missing CSV -> NaNs or placeholders)
    for model_name in model_order_master:
        if model_name not in pre_by_model:
            post_by_q_model[(IDX_RUN, model_name)] = pd.DataFrame()
            missing_val = "-" if model_name in placeholder_models else np.nan
            metadata_rows.append({
                "IDX": IDX_RUN,
                "Question": question_text,
                "Model": model_name,
                "Conversations (rows in CSV)": missing_val,
                "Evaluated speakers (pre filter)": missing_val,
                "failure_rate(regardless of filter_by_lowest)": missing_val,
                "Mean item disagreement rate": missing_val,
                "Evaluated speakers (post filter)": missing_val,
            })
            result_rows.append({
                "IDX": IDX_RUN,
                "Question": question_text,
                "Model": model_name,
                "Categorical pole-consistency": missing_val,
                "Equal Error Rate": missing_val,
                "AUROC": missing_val,
                "Oracle upper bound f1": missing_val,
            })
            continue

        df_wide = wide_by_model[model_name]
        eval_pre = pre_by_model[model_name].copy()

        attempted_slots = 0
        success_slots = 0
        for _, r in df_wide.iterrows():
            for spk in ["a", "b"]:
                avg_s = str(r.get(f"avg_score_{spk}", "")).strip()
                fail_s = str(r.get(f"fail_note_{spk}", "")).strip()
                evid_s = str(r.get(f"evidence_{spk}", "")).strip()

                avg_ok = avg_s not in ("", "nan", "NaN")
                attempted = avg_ok or (fail_s not in ("", "nan", "NaN")) or (evid_s not in ("", "nan", "NaN"))
                if attempted:
                    attempted_slots += 1
                if avg_ok:
                    success_slots += 1

        failure_slots = attempted_slots - success_slots
        failure_rate = (failure_slots / attempted_slots) if attempted_slots > 0 else np.nan
        mean_item_dis = float(pd.to_numeric(eval_pre["disagreement_item_rate"], errors="coerce").dropna().mean()) if len(eval_pre) else np.nan

        eval_post = eval_pre.copy()
        if _exclude_variant_disagree and len(eval_post):
            rate = pd.to_numeric(eval_post.get("disagreement_item_rate", pd.Series(np.nan, index=eval_post.index)), errors="coerce")
            exclude_by_rate = rate >= _disagree_rate_exclude_thresh
            exclude_by_row_level = eval_post.get("row_level_variant_disagreement", pd.Series(False, index=eval_post.index)).fillna(False).astype(bool)
            exclude_mask = exclude_by_rate | exclude_by_row_level
            eval_post = eval_post[~exclude_mask].copy()

        post_by_q_model[(IDX_RUN, model_name)] = eval_post

        metadata_rows.append({
            "IDX": IDX_RUN,
            "Question": question_text,
            "Model": model_name,
            "Conversations (rows in CSV)": int(len(df_wide)),
            "Evaluated speakers (pre filter)": int(len(eval_pre)),
            "failure_rate(regardless of filter_by_lowest)": float(failure_rate),
            "Mean item disagreement rate": float(mean_item_dis) if np.isfinite(mean_item_dis) else np.nan,
            "Evaluated speakers (post filter)": int(len(eval_post)),
        })

        # Metric algorithm is intentionally identical: reuse _compute_perf_metrics from per-question summary cell
        perf = _compute_perf_metrics(eval_post, pos_cat, neg_cat, _use_prob)


        result_rows.append({
            "IDX": IDX_RUN,
            "Question": question_text,
            "Model": model_name,
            **perf,
        })


meta_df_all = pd.DataFrame(metadata_rows)
result_df_all = pd.DataFrame(result_rows)

# Enforce row ordering by parameter order (idx_list first, then model_selection order)
_idx_order_map = {idx: i for i, idx in enumerate(idx_list)}
_model_order_map = {m: i for i, m in enumerate(model_order_master)}

if len(meta_df_all) > 0:
    meta_df_all["__idx_order"] = meta_df_all["IDX"].map(_idx_order_map)
    meta_df_all["__model_order"] = meta_df_all["Model"].map(_model_order_map)
    meta_df_all = meta_df_all.sort_values(["__idx_order", "__model_order"], kind="stable").drop(columns=["__idx_order", "__model_order"]).reset_index(drop=True)
if len(result_df_all) > 0:
    result_df_all["__idx_order"] = result_df_all["IDX"].map(_idx_order_map)
    result_df_all["__model_order"] = result_df_all["Model"].map(_model_order_map)
    result_df_all = result_df_all.sort_values(["__idx_order", "__model_order"], kind="stable").drop(columns=["__idx_order", "__model_order"]).reset_index(drop=True)

# display-only idx remap for tables
if len(meta_df_all) > 0 and "IDX" in meta_df_all.columns:
    meta_df_all["IDX"] = meta_df_all["IDX"].apply(_display_idx_shift)
if len(result_df_all) > 0 and "IDX" in result_df_all.columns:
    result_df_all["IDX"] = result_df_all["IDX"].apply(_display_idx_shift)


if merge_table:
    merge_keys = ["IDX", "Question", "Model"]
    merged_df_all = meta_df_all.merge(result_df_all, on=merge_keys, how="left", sort=False)
    merged_df_show = _select_cols(merged_df_all, merged_col, "merged_col")

    _display_styled_summary_table("Aggregated Merged table (all questions)", merged_df_all, merged_df_show)

else:
    meta_df_show = _select_cols(meta_df_all, meta_table_col, "meta_table_col")
    result_df_show = _select_cols(result_df_all, result_table_col, "result_table_col")

    _display_styled_summary_table("Aggregated Metadata table (all questions)", meta_df_all, meta_df_show)
    _display_styled_summary_table("Aggregated Results table (all questions)", result_df_all, result_df_show)

# # model x # question subplot of binary violins (transposed only for violins)
n_q = len(idx_list)
n_m = len(model_order_master)
if n_q > 0 and n_m > 0:
    fig, axes = plt.subplots(n_m, n_q, figsize=(5 * n_q, 4 * n_m), squeeze=False)
    for r, model_name in enumerate(model_order_master):
        for c, idx in enumerate(idx_list):
            ax = axes[r][c]
            pos_roles, neg_roles = roles_by_q[idx]
            cats_local = cats_by_q.get(idx, [])
            dd = post_by_q_model.get((idx, model_name), pd.DataFrame())
            cats_ticks = [f"{cats_local[0]} + {cats_local[1]}", str(cats_local[2])] if (int(idx) == 9 and len(cats_local) >= 3) else ([str(cats_local[0]), str(cats_local[1])] if len(cats_local) >= 2 else ["POS", "NEG"])
            if dd is None or len(dd) == 0:
                if model_name in placeholder_models:
                    y_refs = plot_refs_by_q_model.get((idx, model_name), ([-1, -0.5, 0, 0.5, 1] if _use_prob else [-2, -1, 0, 1, 2]))
                    y_refs = [float(v) for v in y_refs] if y_refs is not None else []
                    if len(y_refs) == 0:
                        y_refs = [-1, -0.5, 0, 0.5, 1] if _use_prob else [-2, -1, 0, 1, 2]
                    ax.set_xlim(0.5, 2.5)
                    ax.set_xticks([1, 2], labels=[f"POS ({cats_ticks[0]})", f"NEG ({cats_ticks[1]})"])
                    y_lo, y_hi = float(np.min(y_refs)), float(np.max(y_refs))
                    if y_lo == y_hi:
                        y_lo, y_hi = y_lo - 0.5, y_hi + 0.5
                    else:
                        y_pad = 0.05 * (y_hi - y_lo)
                        y_lo, y_hi = y_lo - y_pad, y_hi + y_pad
                    ax.set_ylim(y_lo, y_hi)
                    for y_ref in y_refs:
                        ax.axhline(y_ref, linestyle="--", linewidth=1)
                    if c == 0:
                        ax.set_yticks(y_refs)
                    else:
                        ax.tick_params(axis="y", left=False, labelleft=False)
                        ax.set_yticks([])
                    ax.set_ylabel("")
                    ax.set_xlabel("")
                    if r != (n_m - 1):
                        ax.tick_params(axis="x", bottom=False, labelbottom=False)
                    ax.text(0.5, 0.5, "placeholder", ha="center", va="center", transform=ax.transAxes, color="gray")
                    continue

                if c != 0:
                    ax.set_yticks([])
                ax.axis("off")
                continue

            pos_scores = dd.loc[dd["role"].isin(pos_roles), "score_plot"].astype(float).dropna().values
            neg_scores = dd.loc[dd["role"].isin(neg_roles), "score_plot"].astype(float).dropna().values
            _draw_two_group_violin(
                ax,
                pos_scores,
                neg_scores,
                "",
                "",
                plot_refs_by_q_model[(idx, model_name)],
                cats_ticks,
            )
            if c != 0:
                ax.tick_params(axis="y", left=False, labelleft=False)
                ax.set_ylabel("")
            if r != (n_m - 1):
                ax.tick_params(axis="x", bottom=False, labelbottom=False)
                ax.set_xlabel("")
    for c, idx in enumerate(idx_list):
        axes[0][c].set_title(f"IDX {_display_idx_shift(idx)}", fontsize=12, pad=10)
    plt.tight_layout(rect=(0.06, 0, 1, 1))
    for r, model_name in enumerate(model_order_master):
        pos = axes[r][0].get_position()
        y_mid = 0.5 * (pos.y0 + pos.y1)
        x_left = max(0.001, pos.x0 - 0.015)
        fig.text(x_left, y_mid, model_name, ha="center", va="center", rotation=90, fontsize=13)
    plt.show()

# 1 x # question subplot of all ROC plots
if n_q > 0:
    fig, axes = plt.subplots(1, n_q, figsize=(6 * n_q, 5), squeeze=False)
    for i, idx in enumerate(idx_list):
        ax = axes[0][i]
        pos_roles, neg_roles = roles_by_q[idx]
        any_curve = False

        for model_name in model_order_master:
            dd = post_by_q_model.get((idx, model_name), pd.DataFrame())
            if dd is None or len(dd) == 0:
                continue

            merged = dd[dd["role"].isin(set(pos_roles) | set(neg_roles))].copy()
            merged = merged.dropna(subset=["score_used"])
            if len(merged) == 0:
                continue

            merged["y_true"] = merged["role"].isin(pos_roles).astype(int)
            y_true = merged["y_true"].values
            y_score = merged["score_used"].astype(float).values
            if len(np.unique(y_true)) < 2:
                continue

            try:
                from sklearn.metrics import roc_curve, roc_auc_score
                fpr, tpr, _ = roc_curve(y_true, y_score)
                auc_val = float(roc_auc_score(y_true, y_score))
            except Exception:
                pos = y_score[y_true == 1]
                neg = y_score[y_true == 0]
                if len(pos) == 0 or len(neg) == 0:
                    continue
                u = mannwhitneyu(pos, neg, alternative="two-sided")
                auc_val = float(u.statistic / (len(pos) * len(neg)))
                thr = np.sort(np.unique(y_score))
                fpr, tpr = [], []
                for t in thr:
                    pred = (y_score >= t).astype(int)
                    tp = ((pred == 1) & (y_true == 1)).sum()
                    fp = ((pred == 1) & (y_true == 0)).sum()
                    fn = ((pred == 0) & (y_true == 1)).sum()
                    tn = ((pred == 0) & (y_true == 0)).sum()
                    tpr.append(tp / (tp + fn) if (tp + fn) else 0.0)
                    fpr.append(fp / (fp + tn) if (fp + tn) else 0.0)
                fpr = np.array(fpr)
                tpr = np.array(tpr)

            any_curve = True
            (line,) = ax.plot(fpr, tpr, linewidth=2, label=f"{model_name} (AUC={auc_val:.3f})")
            ax.fill_between(fpr, tpr, 0, alpha=0.15, color=line.get_color())

        ax.plot([0, 1], [0, 1], linestyle="--", linewidth=1, color="gray")
        ax.set_title(f"IDX {_display_idx_shift(idx)}: ROC across models")
        ax.set_xlabel("False Positive Rate")
        if i == 0:
            ax.set_ylabel("True Positive Rate")
        if any_curve:
            ax.legend(loc="lower right")
        else:
            ax.text(0.5, 0.5, "[WARN] no valid ROC curves", ha="center", va="center", transform=ax.transAxes)

    plt.tight_layout()
    plt.show()

In [ ]:
# Mirrored half-violin summary plot (model x [POS, NEG])
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from scipy.stats import gaussian_kde

_required = ["idx_list", "model_order_master", "post_by_q_model", "roles_by_q", "cats_by_q"]
_missing = [k for k in _required if k not in globals()]
if _missing:
    raise RuntimeError(f"Missing required variables from previous summary cell: {_missing}")


def _clean_signed_prob(vals):
    arr = np.asarray(vals, dtype=float)
    arr = arr[np.isfinite(arr)]
    if arr.size == 0:
        return arr
    # Keep y-domain fixed to signed probability range.
    return np.clip(arr, -1.0, 1.0)


def _kde_peak(vals, y_grid):
    vals = _clean_signed_prob(vals)
    if vals.size == 0:
        return 0.0

    if vals.size >= 2 and np.std(vals) > 0 and np.unique(vals).size > 1:
        try:
            d = gaussian_kde(vals)(y_grid)
        except np.linalg.LinAlgError:
            bw = max((float(y_grid[-1]) - float(y_grid[0])) / 40.0, 1e-6)
            d = np.exp(-0.5 * ((y_grid - float(np.mean(vals))) / bw) ** 2)
    elif vals.size == 1:
        bw = max((float(y_grid[-1]) - float(y_grid[0])) / 40.0, 1e-6)
        d = np.exp(-0.5 * ((y_grid - float(vals[0])) / bw) ** 2)
    else:
        d = np.zeros_like(y_grid)

    vmin, vmax = float(np.min(vals)), float(np.max(vals))
    d[(y_grid < vmin) | (y_grid > vmax)] = 0.0
    return float(np.max(d)) if d.size else 0.0


# Fixed 9-color stance palette (consistent across all models and POS/NEG columns)
stance_palette = [
    "#1b9e77", "#d95f02", "#7570b3", "#e7298a", "#66a61e",
    "#e6ab02", "#a6761d", "#1f78b4", "#b2df8a",
]

idx_list_local = list(idx_list)
display_idx_shift_map_local = {7: 6, 8: 7, 9: 8}  # display-only remap

def _display_idx_shift_local(v):
    try:
        i = int(v)
    except Exception:
        return v
    return display_idx_shift_map_local.get(i, i)

model_list_local = list(model_order_master)
n_q = len(idx_list_local)
n_m = len(model_list_local)

if n_q == 0 or n_m == 0:
    print("[WARN] Empty idx_list or model_order_master; skipping mirrored half-violin plot.")
else:
    if n_q > len(stance_palette):
        cmap = plt.get_cmap("tab20")
        stance_palette = [cmap(i % 20) for i in range(n_q)]

    stance_color = {idx: stance_palette[i] for i, idx in enumerate(idx_list_local)}

    # Cache per-model, per-stance values.
    pos_vals_by_model_stance = {}
    neg_vals_by_model_stance = {}
    for model_name in model_list_local:
        for idx in idx_list_local:
            dd = post_by_q_model.get((idx, model_name), pd.DataFrame())
            pos_roles, neg_roles = roles_by_q[idx]
            if dd is None or len(dd) == 0 or "score_plot" not in dd.columns:
                pos_vals = np.array([], dtype=float)
                neg_vals = np.array([], dtype=float)
            else:
                pos_vals = _clean_signed_prob(
                    dd.loc[dd["role"].isin(pos_roles), "score_plot"].astype(float).values
                )
                neg_vals = _clean_signed_prob(
                    dd.loc[dd["role"].isin(neg_roles), "score_plot"].astype(float).values
                )
            pos_vals_by_model_stance[(model_name, idx)] = pos_vals
            neg_vals_by_model_stance[(model_name, idx)] = neg_vals

    # Width normalization: one shared ruler per question (Qk),
    # used across all models and both POS/NEG columns.
    # Robust version: drop one highest peak then take percentile to reduce single outlier impact.
    y_grid = np.linspace(-1.0, 1.0, 256)
    question_peak_by_idx = {}
    question_peak_percentile = 99
    drop_top_peak_per_question = True
    for idx in idx_list_local:
        peaks = []
        for model_name in model_list_local:
            p_peak = _kde_peak(pos_vals_by_model_stance[(model_name, idx)], y_grid)
            n_peak = _kde_peak(neg_vals_by_model_stance[(model_name, idx)], y_grid)
            if np.isfinite(p_peak) and p_peak > 0:
                peaks.append(float(p_peak))
            if np.isfinite(n_peak) and n_peak > 0:
                peaks.append(float(n_peak))
        if peaks:
            peak_arr = np.asarray(peaks, dtype=float)
            if drop_top_peak_per_question and peak_arr.size >= 4:
                peak_arr = np.sort(peak_arr)[:-1]
            q_peak = float(np.percentile(peak_arr, question_peak_percentile))
        else:
            q_peak = 0.0
        question_peak_by_idx[idx] = q_peak if (np.isfinite(q_peak) and q_peak > 0) else 1.0

    pos_labels = []
    neg_labels = []
    for idx in idx_list_local:
        cats = cats_by_q.get(idx, [])
        if int(idx) == 9 and len(cats) >= 3:
            pos_labels.append(f"{cats[0]} + {cats[1]}")
            neg_labels.append(str(cats[2]))
        else:
            pos_labels.append(str(cats[0]) if len(cats) >= 1 else f"IDX {_display_idx_shift_local(idx)} POS")
            neg_labels.append(str(cats[1]) if len(cats) >= 2 else f"IDX {_display_idx_shift_local(idx)} NEG")
    q_labels = [f"Q{_display_idx_shift_local(idx)}" for idx in idx_list_local]

    # Make the chart taller than wide and visually denser on x.

    fig_w = max(15.0, 1.50 * n_q * 2)
    fig_h = fig_w * 0.45
    # fig_h = max(fig_w * 1, 3.4 * n_m)

    # Compressed stance spacing (closer violins).
    x_step = 0.32
    x_positions = 1.0 + np.arange(n_q, dtype=float) * x_step
    visible_edge_pad = 0.08
    hidden_edge_pad = 0.08

    max_half_width = 0.35
    # Dynamic geometric cap so spacing params stay valid after density capping.
    # Keeps at least `hidden_edge_pad` gap between neighboring visible halves.
    cap_relax = 0.06
    effective_half_width = float(max(1e-6, min(max_half_width, x_step - hidden_edge_pad+ cap_relax)))
    y_ticks = [-1.0, 0.0, 1.0]
    y_guides = [-0.5, 0.0, 0.5]

    # Typography controls (adjust these manually)
    x_subtitle_fontsize = 40
    model_name_fontsize = 50
    x_tick_fontsize = 40
    y_tick_fontsize = 32
    column_title_fontsize = 54
    column_title_pad = 26


    def _draw_group_on_axis(ax, model_name, grp_key, alpha):
        for x, idx in zip(x_positions, idx_list_local):
            vals = (
                pos_vals_by_model_stance[(model_name, idx)]
                if grp_key == "POS"
                else neg_vals_by_model_stance[(model_name, idx)]
            )
            if vals.size == 0:
                continue

            vals_arr = np.asarray(vals, dtype=float)
            vals_arr = vals_arr[np.isfinite(vals_arr)]
            if vals_arr.size == 0:
                continue

            if vals_arr.size >= 2 and np.std(vals_arr) > 0 and np.unique(vals_arr).size > 1:
                try:
                    dens = gaussian_kde(vals_arr)(y_grid)
                except np.linalg.LinAlgError:
                    bw = max((float(y_grid[-1]) - float(y_grid[0])) / 40.0, 1e-6)
                    dens = np.exp(-0.5 * ((y_grid - float(np.mean(vals_arr))) / bw) ** 2)
            elif vals_arr.size == 1:
                bw = max((float(y_grid[-1]) - float(y_grid[0])) / 40.0, 1e-6)
                dens = np.exp(-0.5 * ((y_grid - float(vals_arr[0])) / bw) ** 2)
            else:
                dens = np.zeros_like(y_grid)

            vmin, vmax = float(np.min(vals_arr)), float(np.max(vals_arr))
            dens[(y_grid < vmin) | (y_grid > vmax)] = 0.0

            local_peak = float(np.max(dens)) if dens.size else 0.0
            q_peak = question_peak_by_idx[idx]
            width_scale = local_peak / q_peak if q_peak > 0 else 0.0
            width_scale = float(np.clip(width_scale, 0.0, 1.0))
            if width_scale <= 0:
                continue

            half_w = (np.minimum(dens / q_peak, 1.0) * effective_half_width) if q_peak > 0 else np.zeros_like(dens)
            color = stance_color[idx]

            # True half-violin rendering (no hidden half kept in geometry).
            if grp_key == "POS":
                x0 = x - half_w
                x1 = np.full_like(y_grid, x)
            else:
                x0 = np.full_like(y_grid, x)
                x1 = x + half_w
            ax.fill_betweenx(y_grid, x0, x1, facecolor=color, edgecolor=color, alpha=alpha, linewidth=0.8)

            # Stance-colored median bar on the visible half only.
            med = float(np.median(vals_arr))
            med_half = effective_half_width * width_scale
            if med_half > 0:
                if grp_key == "POS":
                    ax.plot([x - med_half, x], [med, med], color=color, linewidth=1.8, zorder=3)
                else:
                    ax.plot([x, x + med_half], [med, med], color=color, linewidth=1.8, zorder=3)

    def _style_axis(ax, x_labels, grp_key):
        for y in y_guides:
            ax.axhline(
                y,
                linestyle="--",
                linewidth=1.0,
                color="0.65",
                zorder=0,
            )
        if grp_key == "POS":
            x_left = x_positions[0] - effective_half_width - visible_edge_pad
            x_right = x_positions[-1] + hidden_edge_pad
        else:
            x_left = x_positions[0] - hidden_edge_pad
            x_right = x_positions[-1] + effective_half_width + visible_edge_pad
        ax.set_xlim(float(x_left), float(x_right))
        ax.set_ylim(-1.0, 1.0)
        ax.set_yticks(y_ticks)
        ax.set_xticks(x_positions)
        ax.set_xticklabels(x_labels, rotation=0, ha="right", fontsize=x_subtitle_fontsize)

    fig, axes = plt.subplots(n_m, 2, figsize=(fig_w, fig_h), sharey=True, squeeze=False)

    for r, model_name in enumerate(model_list_local):
        for c, (grp, grp_key, x_labels) in enumerate([
            ("Positive pole", "POS", q_labels),
            ("Negative pole", "NEG", q_labels),
        ]):
            ax = axes[r, c]

            _draw_group_on_axis(ax, model_name, grp_key, alpha=0.45)

            for y in y_guides:
                ax.axhline(
                    y,
                    linestyle="--",
                    linewidth=1.0,
                    color="0.65",
                    zorder=0,
                )

            if grp_key == "POS":
                x_left = x_positions[0] - effective_half_width - visible_edge_pad
                x_right = x_positions[-1] + hidden_edge_pad
            else:
                x_left = x_positions[0] - hidden_edge_pad
                x_right = x_positions[-1] + effective_half_width + visible_edge_pad
            ax.set_xlim(float(x_left), float(x_right))
            ax.set_ylim(-1.0, 1.0)
            ax.set_yticks(y_ticks)
            ax.tick_params(axis="y", labelsize=y_tick_fontsize)

            # Only last row shows x ticks/labels.
            if r == (n_m - 1):
                ax.set_xticks(x_positions)
                ax.tick_params(axis="x", labelsize=x_tick_fontsize)
                ax.set_xticklabels(x_labels, ha="right", fontsize=x_subtitle_fontsize)
            else:
                ax.set_xticks([])
                ax.tick_params(axis="x", bottom=False, labelbottom=False)

            if c == 0:
                ax.set_ylabel(f"{model_name}", fontsize=model_name_fontsize)
            else:
                ax.set_ylabel("")

            if r == 0:
                ax.set_title(f"{grp}", fontsize=column_title_fontsize, pad=column_title_pad)

    plt.tight_layout()
    plt.show()